<div align="center">

# Hafta 10: Klinik karar destek sistemleri

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/lab.html) · [PDF ders notu](Hafta10_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-10/NB10_klinik_karar_destegi.ipynb)

</div>

**Bu defter iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı defter, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. 3. bölümde 14.883 dakikanın beş sayısı yaklaşık on saniyede hesaplanır; model de sıradan bir dizüstü bilgisayarda yaklaşık bir saniyede eğitilir. 8. bölüm prototipin iki dosyasını yazar ve bu bilgisayarda küçük bir tahmin servisi başlatır; web arayüzü ardından Anaconda Prompt'ta başlatılır. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. İkinci hücre ayrıca MIT-BIH Atriyal Fibrilasyon Veri Tabanı'nın 25 kaydının işaretleme dosyalarını ve kayıtlarından üçünün sinyallerini PhysioNet'ten yaklaşık 86 MB olarak bir ile üç dakikada indirir ve her dosyayı PhysioNet'in parmak izi listesiyle denetler; zaten bulunan dosyalar atlanır. Telemetri servisinin hastaları ders paketiyle birlikte gelir.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import hashlib
import json
import pathlib
import re
import urllib.request

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from vclinic import afdb, holter
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 10, 0):
    raise ImportError("vclinic ders paketi bu defterden daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
afdb.download()
print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["streamlit"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 10.1": (133, 0),
    "Alıştırma 10.2": (3, 0),
    "Alıştırma 10.3": (2, 0),
    "Alıştırma A": (12.0, 0.05),
    "Alıştırma B": (1490, 0),
    "Alıştırma C": (1.98, 0.01),
    "Alıştırma D": (89, 3),
    "Alıştırma E": (9.1, 0.05),
    "Alıştırma F": (8, 0),
    "Alıştırma G": (8, 0),
    "Alıştırma H": (1429, 0),
    "Alıştırma I": (60, 0),
    "Alıştırma J": (0.924, 0.005),
    "Adım 1.1": (0.222, 0.005),
    "Adım 1.2": (0.37, 0.02),
    "Adım 1.3": (22, 3),
    "Adım 2.1": (0.985, 0.005),
    "Adım 2.2": (15.5, 0.05),
    "Adım 2.3": (222, 0),
    "Adım 3.1": (76, 0),
    "Adım 3.2": (69, 6),
    "Adım 3.3": (0.924, 0.01),
})
print("Beklenen cevaplar hazır.")

In [ ]:
#@title Alıştırma promptlarının tabloları (bu hücreyi çalıştırın; Python yolunda okunması gerekmez)
def alistirma_tablosu(anahtar):
    """Bir alıştırma promptunun "veri" tablosu; haftanın verilerinden kurulur."""
    g = globals()
    k = anahtar
    need = lambda *names: [g[n] for n in names] if all(n in g for n in names) else (_ for _ in ()).throw(
        RuntimeError("Önce bu alıştırmadan önceki bölümleri çalıştırın."))
    sets = {"training": "eğitim", "validation": "doğrulama", "test": "test"}
    if k == "A":
        hours, = need("hours")
        return pd.DataFrame({"kayıt": hours.index, "küme": hours["set"].map(sets).to_numpy(),
                             "AF (saat)": hours["(AFIB"].round(2).to_numpy(), "atriyal flatter (saat)": hours["(AFL"].round(2).to_numpy(),
                             "kavşak ritmi (saat)": hours["(J"].round(2).to_numpy(), "öteki ritimler (saat)": hours["(N"].round(2).to_numpy()})
    if k == "B":
        minutes, = need("minutes")
        return pd.DataFrame({"kayıt": minutes["record"], "küme": minutes["set"].map(sets), "başlangıç (s)": minutes["start_s"].round(2),
                             "AF": minutes["af"]}).reset_index(drop=True)
    if k == "C":
        minutes, = need("minutes")
        m = minutes[minutes["set"] == "training"]
        return pd.DataFrame({"kayıt": m["record"], "AF": m["af"], "RR aralıkları (s)": [list(np.round(x, 3)) for x in m["rr"]]}).reset_index(drop=True)
    if k in ("D", "F"):
        validation, p_val = need("validation", "p_val")
        return pd.DataFrame({"kayıt": validation["record"], "başlangıç (s)": validation["start_s"].round(2), "AF": validation["af"],
                             "olasılık": np.round(p_val, 4)}).reset_index(drop=True)
    if k == "E":
        validation, model = need("validation", "model")
        part = validation[(validation["record"] == "08219") & (validation["af"] == 1)]
        x = part.loc[part.index[af_probability(model, part).argmax()]]
        names = {"hr": "kalp hızı", "cv": "CV", "rmssd": "RMSSD/RR", "pnn50": "pNN50"}
        return pd.DataFrame({"sayı": [names[f] for f in model["features"]] + ["sabit terim"],
                             "dakikadaki değer": [round(float(x[f]), 4) for f in model["features"]] + [np.nan],
                             "önce logaritma": ["evet" if f in ("cv", "rmssd") else "hayır" for f in model["features"]] + [""],
                             "ortalama, eğitim": [model["mean"][f] for f in model["features"]] + [np.nan],
                             "standart sapma, eğitim": [model["sd"][f] for f in model["features"]] + [np.nan],
                             "katsayı": [model["coef"][f] for f in model["features"]] + [model["intercept"]]})
    if k == "G":
        final, = need("final")
        return pd.DataFrame({"rapor": ["VC-0128"], "paket (JSON)": [json.dumps(final, ensure_ascii=False)]})
    if k == "H":
        rec_0128, = need("rec_0128")
        return pd.DataFrame({"vuru zamanı (s)": np.round(rec_0128["detected_s"], 3)})
    if k == "I":
        ward_minutes, = need("ward_minutes")
        return pd.DataFrame({"hasta": ward_minutes["record"], "başlangıç (sa)": (ward_minutes["start_s"] / 3600).round(4),
                             "AF": ward_minutes["af"], "olasılık": ward_minutes["p10"].round(4)}).reset_index(drop=True)
    if k == "J":
        test, model_11 = need("test", "model_11")
        return pd.DataFrame({"kayıt": test["record"], "başlangıç (s)": test["start_s"].round(2), "AF": test["af"],
                             "olasılık": np.round(af_probability(model_11, test), 4)}).reset_index(drop=True)
    raise KeyError(k)
print("Alıştırma promptlarının tabloları hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Bu kısım kod yazmayan katılımcılar içindir. Burada hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptun altındaki veri hücresini çalıştırın. Hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve tablonun ilk ve son satırlarını gösterir.
3. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
4. Promptu YZ asistanına yapıştırın ve gönderin.
5. Asistanın verdiği kodu kopyalayın, veri hücresinin altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
6. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
7. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**"veri" ve "cevap".** 1. haftada olduğu gibi her adımın veri hücresi, promptun tarif ettiği "veri" tablosunu hazırlar. Asistan sonucu "cevap" adıyla saklar; kontrol hücresi de sonucu bu addan okur.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter defterinde çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-10/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: Prototipin AF modeli

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-4. bölümler*

Bu durak oturumun mini projesidir. AFDB'nin 25 kaydının dakikaları RR aralıklarından elde edilen dört sayıyla tarif edilir; bir lojistik regresyon AF'yi eğitim kayıtlarından öğrenir, eşiği doğrulama kayıtlarında seçilir ve test kayıtları her kaydın AF yükünü ve alarm ataklarını verir.

**Durağın sonunda:** Eğitim kayıtlarındaki AF dakikalarının ortanca CV değerini (0,222), modelin eşiğini (0,37) ve test kayıtlarındaki yanlış alarm ataklarını (22) bulacaksınız.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test"}


def _course():
    """AFDB'nin 25 kaydının ve telemetri servisindeki dört hastanın dakikaları; RR aralıkları, beş sayısı ve 1.0 sürümünün olasılıklarıyla birlikte (bir kez hesaplanır, yaklaşık yirmi saniye sürer)."""
    if "_course_cache" in globals():
        return _course_cache
    SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
    episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
    rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                            for record in afdb.RECORDS for start, end, name in episodes[record]])
    hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
    hours["set"] = hours.index.map(SPLIT)

    WINDOW_S = 60


    def minutes_of(beat_s, episodes=None, record=""):
        """Bir kaydın bir dakikalık pencereleri: Her dakikada biten RR aralıkları ve ritim biliniyorsa dakikanın AF içinde geçen payı."""
        rr = np.diff(beat_s)
        rr_end = beat_s[1:]
        rows = []
        for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
            x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
            if len(x) < 20 or x.max() > 3.0:
                continue
            row = {"record": record, "start_s": float(start), "rr": x}
            if episodes is not None:
                af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
                row["af_share"] = af_s / WINDOW_S
            rows.append(row)
        return rows

    minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
    minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
    minutes["set"] = minutes["record"].map(SPLIT)


    def sample_entropy(rr, m=1, r=0.03):
        """Örneklem entropisi: İlk m aralığı birbirine benzeyen, yani en fazla r saniye farklı olan iki dizinin m + 1 aralıkta da benzer olma olasılığının logaritmasının eksi işaretlisi."""
        n = len(rr)

        def pairs(length):
            t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
            alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
            return (alike.sum() - len(t)) / 2

        b, a = pairs(m), pairs(m + 1)
        return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


    def features(rr):
        """Bir dakikanın beş sayısı: Kalp hızı, değişim katsayısı, ortalama RR'ye bölünmüş RMSSD, pNN50 ve örneklem entropisi."""
        d = np.diff(rr)
        return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
                "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

    minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

    V1_0 = ["hr", "cv", "rmssd", "pnn50"]
    LOGGED = ["cv", "rmssd"]


    def design(table, names):
        """Modelin değişkenleri: CV'nin ve RMSSD'nin logaritması, ötekiler olduğu gibi."""
        X = table[names].astype(float).copy()
        for name in names:
            if name in LOGGED:
                X[name] = np.log(X[name])
        return X


    def fit_model(names, version, training):
        """Standartlaştırılmış değişkenlerle bir lojistik regresyon; JSON'un saklayabileceği düz bir sözlük olarak tutulur."""
        X = design(training, names)
        mean, sd = X.mean(), X.std()
        lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
        return {"name": "RR aralıklarından AF", "version": version, "features": names,
                "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
                "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


    def af_probability(model, table):
        """Bir tablonun her dakikası için AF olasılığı."""
        names = model["features"]
        z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
        w = np.array([model["coef"][name] for name in names])
        return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


    def choose_threshold(p, y, sensitivity=0.95):
        """AF dakikalarının istenen payını hâlâ bulan, 0,01 adımlı en yüksek eşik."""
        grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
        return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


    def alarm_episodes(alarm):
        """Art arda gelen alarm dakikalarının dizileri: (ilk, son) konumları."""
        runs, first = [], None
        for k, a in enumerate(alarm):
            if a and first is None:
                first = k
            if not a and first is not None:
                runs.append((first, k - 1))
                first = None
        if first is not None:
            runs.append((first, len(alarm) - 1))
        return runs


    def by_record(table, p, threshold):
        """Her kayıt için işaretlemenin ve modelin AF yükü, alarm atakları ve bunların yanlış olanları."""
        rows = []
        for record, part in table.assign(p=p).groupby("record", sort=True):
            alarm = (part["p"] >= threshold).to_numpy()
            runs = alarm_episodes(alarm)
            false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
            rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                         "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
        return pd.DataFrame(rows).set_index("record")

    training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
    model = fit_model(V1_0, "1.0.0", training)
    p_val = af_probability(model, validation)
    model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())


    def contributions(model, row):
        """Her değişkenin bir dakikadaki AF log-odds değerine kattığı: Katsayısı çarpı z-skoru."""
        x = design(pd.DataFrame([row]), model["features"]).iloc[0]
        z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
        return {name: model["coef"][name] * z[name] for name in model["features"]}


    def predict(request, model, digest):
        """Tahmin servisi: Bir kaydın vuru zamanlarını içeren istek girer, sonuç çıkar; ikisi de JSON'un taşıyabileceği sözlüklerdir."""
        table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
        table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
        p = af_probability(model, table)
        alarm = p >= model["threshold"]
        runs = alarm_episodes(alarm)
        return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
                "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
                "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

    ward = holter.ward()
    ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                                 for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
    ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
    ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))
    minutes["p10"] = af_probability(model, minutes)
    ward_minutes["p10"] = af_probability(model, ward_minutes)
    digest = hashlib.sha256(json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    globals()["_course_cache"] = dict(minutes=minutes, model=model, digest=digest, ward=ward, ward_minutes=ward_minutes,
                                      contributions=contributions, predict=predict, design=design, by_record=by_record)
    return _course_cache


def _minutes_table(columns, sets=("training", "validation", "test")):
    """Seçilen sütunlarla AFDB dakikalarının bir tablosu."""
    m = _course()["minutes"]
    m = m[m["set"].isin(sets)]
    cols = {"kayıt": m["record"], "küme": m["set"].map(_SETS), "başlangıç (s)": m["start_s"].round(2),
            "AF": m["af"], "AF payı": m["af_share"].round(3), "RR aralıkları (s)": [list(np.round(x, 3)) for x in m["rr"]],
            "kalp hızı": m["hr"].round(2), "CV": m["cv"].round(4), "RMSSD/RR": m["rmssd"].round(4), "pNN50": m["pnn50"].round(4),
            "olasılık": m["p10"].round(4)}
    return pd.DataFrame({k: v for k, v in cols.items() if k in columns}).reset_index(drop=True)


_c = _course()
print(len(_c["minutes"]), "dakika AFDB'nin 25 kaydında")
print("1. durak hazır.")

### Adım 1.1: Bir dakikanın dört sayısı

**Amaç.** Bu adımda AFDB'nin her dakikası RR aralıklarından elde edilen dört sayıyla tarif edilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir telemetri servisinin kardiyoloğu, vuruların zamanlarından olası AF dakikalarını işaretleyen bir karar destek prototipi istiyor. İlk adım, her dakikayı RR aralıklarının ne kadar düzensiz olduğunu ölçen sayılarla tarif etmektir.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda MIT-BIH Atriyal Fibrilasyon Veri Tabanı'ndaki (AFDB) çoğu paroksismal atriyal fibrilasyonu (AF) olan kişilerin yaklaşık on saatlik 25 Holter kaydından elde edilen bir dakikalık pencereler bulunur. Her satır bir dakikadır: "kayıt" kaydı, "küme" kaydın kümesini ("eğitim", 15 kayıt; "doğrulama", 5 kayıt; "test", 5 kayıt), "başlangıç (s)" ise dakikanın kaydın başından itibaren saniye cinsinden başlangıcını verir. Dakikanın yarısından fazlası uzmanların işaretlediği AF içindeyse "AF" 1, değilse 0 olur. "RR aralıkları (s)" sütununun her hücresi, dakikanın RR aralıklarını saniye cinsinden ve sırasıyla, 20 ya da daha fazla sayıdan oluşan bir liste olarak tutar.  
> Görev: Her dakika için RR aralıklarından dört sayı hesapla: Kalp hızı, yani 60'ın ortalama aralığa bölümü; değişim katsayısı, yani aralıkların standart sapmasının (n − 1 ile) ortalamalarına bölümü; RMSSD'nin ortalama aralığa bölümü (RMSSD, ardışık aralıklar arasındaki farkların karelerinin ortalamasının kareköküdür); ve pNN50, yani 0,05 saniyeden büyük ardışık farkların oranı. Ardından eğitim kayıtlarında AF olan ve olmayan dakikaların dört sayısını karşılaştır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eğitim dakikalarında her sayının AF olan ve olmayan dakikalardaki ortancasını yazdır; AF olan eğitim dakikalarının ortanca CV değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her dakikanın kalp hızı dakikada 20 ile 250 arasında, pNN50 değeri ise 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _minutes_table(["kayıt", "küme", "başlangıç (s)", "AF", "RR aralıkları (s)"])
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "cevap", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _minutes_table(["kayıt", "küme", "başlangıç (s)", "AF", "RR aralıkları (s)"])
d = veri


def dort(rr):
    rr = np.asarray(rr, dtype=float)
    fark = np.diff(rr)
    return pd.Series({"kalp hızı": 60 / rr.mean(), "CV": rr.std(ddof=1) / rr.mean(),
                      "RMSSD/RR": np.sqrt(np.mean(fark ** 2)) / rr.mean(), "pNN50": np.mean(np.abs(fark) > 0.05)})


sayilar = d["RR aralıkları (s)"].apply(dort)
tablo = pd.concat([d[["kayıt", "küme", "AF"]], sayilar], axis=1)
egitim = tablo[tablo["küme"] == "eğitim"]
print(egitim.groupby("AF")[["kalp hızı", "CV", "RMSSD/RR", "pNN50"]].median().round(3))
cevap = round(float(egitim.loc[egitim["AF"] == 1, "CV"].median()), 3)
teach.check_step("Adım 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** AF dakikalarının ortanca CV değeri 0,222 düzeyindedir ve öteki dakikalarınkinin birkaç katıdır: AF'de RR aralıkları vurudan vuruya çok değişir ve dört sayı bunu yakalar.

### Adım 1.2: Model ve eşiği

**Amaç.** Bu adımda AF modelinin 1.0 sürümü eğitim kayıtlarında kurulur ve eşiği doğrulama kayıtlarında seçilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog prototipin olabildiğince az AF dakikası kaçırmasını istiyor: AF dakikalarının en az yüzde 95'i alarm vermeli. Yanlış alarmlar bunun bedelidir; test kayıtları da son değerlendirmeye kadar hiç kullanılmamalıdır.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda MIT-BIH Atriyal Fibrilasyon Veri Tabanı'ndaki (AFDB) çoğu paroksismal atriyal fibrilasyonu (AF) olan kişilerin yaklaşık on saatlik 25 Holter kaydından elde edilen bir dakikalık pencereler bulunur. Her satır bir dakikadır: "kayıt" kaydı, "küme" kaydın kümesini ("eğitim", 15 kayıt; "doğrulama", 5 kayıt; "test", 5 kayıt), "başlangıç (s)" ise dakikanın saniye cinsinden başlangıcını verir. Dakikanın yarısından fazlası uzmanların işaretlediği AF içindeyse "AF" 1, değilse 0 olur. Dört sayı dakikanın RR aralıklarını tarif eder: "kalp hızı" (dakikada), "CV" (aralıkların standart sapmasının ortalamalarına bölümü), "RMSSD/RR" (ardışık farkların karelerinin ortalamasının karekökünün ortalama aralığa bölümü) ve "pNN50" (50 ms'den büyük ardışık farkların oranı).  
> Görev: Yalnızca eğitim dakikalarında AF için bir lojistik regresyon kur; en fazla 3.000 yineleme kullan. Girdileri kalp hızı, CV'nin doğal logaritması, RMSSD/RR'nin doğal logaritması ve pNN50'dir; her biri eğitim dakikalarının ortalaması ve standart sapmasıyla bir z-skoruna dönüştürülür. Her doğrulama dakikası için AF olasılığını ve doğrulama dakikalarındaki ROC eğrisi altındaki alanı (AUC) hesapla. Eşiği yalnızca doğrulama dakikalarında seç: 0,05, 0,06, 0,07 ve bu biçimde 0,95'e kadar giden değerler arasından, AF olan doğrulama dakikalarının en az yüzde 95'inin olasılığının ona eşit ya da ondan büyük olduğu en yüksek değeri al.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Doğrulama AUC değerini, eşiği ve modelin katsayılarını yazdır; eşiği iki ondalık basamakla "cevap" adıyla sakla.  
> Test: Hiçbir test dakikası kullanılmamalı ve AF olan doğrulama dakikalarının eşikte ya da üstünde kalan payı en az 0,95 olmalı.  

In [ ]:
#@title Adım 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _minutes_table(["kayıt", "küme", "başlangıç (s)", "AF", "kalp hızı", "CV", "RMSSD/RR", "pNN50"])
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "cevap", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _minutes_table(["kayıt", "küme", "başlangıç (s)", "AF", "kalp hızı", "CV", "RMSSD/RR", "pNN50"])
d = veri
X = pd.DataFrame({"kalp hızı": d["kalp hızı"], "log CV": np.log(d["CV"]),
                    "log RMSSD/RR": np.log(d["RMSSD/RR"]), "pNN50": d["pNN50"]})
eg_ = (d["küme"] == "eğitim").to_numpy()
do_ = (d["küme"] == "doğrulama").to_numpy()
ort, ss = X[eg_].mean(), X[eg_].std()
lr = LogisticRegression(max_iter=3000).fit((X[eg_] - ort) / ss, d["AF"][eg_])
p = lr.predict_proba((X[do_] - ort) / ss)[:, 1]
y = d["AF"].to_numpy()[do_]
izgara = np.round(np.arange(0.05, 0.96, 0.01), 2)
esik = max(t for t in izgara if (p[y == 1] >= t).mean() >= 0.95)
print("doğrulama AUC", round(roc_auc_score(y, p), 3), "| eşik", esik)
print("katsayılar", dict(zip(X.columns, np.round(lr.coef_[0], 3))), "| sabit terim", round(lr.intercept_[0], 3))
cevap = round(float(esik), 2)
teach.check_step("Adım 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** Eşik 0,37 olarak bulunur. Bu eşik doğrulama kayıtlarında en az yüzde 95 duyarlılığı korur; kaç yanlış alarma mal olduğunu bir sonraki adım ve kullanıcı testi gösterir.

### Adım 1.3: Test kayıtlarının AF yükü ve alarm atakları

**Amaç.** Bu adımda test dakikalarının olasılıkları her test kaydı için alarm ataklarına ve bir AF yüküne dönüştürülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog tek tek dakikaları değil, art arda gelen alarm dakikalarını inceler ve her kaydın AF yükünü raporun vereceği biçimde görmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda MIT-BIH Atriyal Fibrilasyon Veri Tabanı'nın 5 test kaydının, yani çoğu paroksismal atriyal fibrilasyonu (AF) olan kişilerin Holter kayıtlarının bir dakikalık pencereleri bulunur. Her satır bir dakikadır: "kayıt" kaydı, "başlangıç (s)" ise dakikanın saniye cinsinden başlangıcını verir. Dakikanın yarısından fazlası uzmanların işaretlediği AF içindeyse "AF" 1 olur; "AF payı" dakikanın AF içinde geçen payıdır ve 0 ile 1 arasında değer alır; "olasılık" ise dersin bir modelinin dakikaya verdiği AF olasılığıdır. Model, olasılığı en az 0,37 olan bir dakika için alarm verir.  
> Görev: Her kayıt içinde dakikaları başlangıçlarına göre sırala. Bir alarm atağı, alarm veren art arda dakikalardan oluşan bir dizidir; içinde AF olan hiçbir dakika bulunmayan atak ise bir yanlış alarm atağıdır. Her kayıt için işaretlemenin AF yükünü, yani dakikalarının ortalama AF payının 100 katını; modelin AF yükünü, yani alarm veren dakikalarının payının 100 katını; alarm ataklarının sayısını ve yanlış alarm ataklarının sayısını hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her kayıt için bir satırdan oluşan bir tablo göster; test kayıtlarındaki toplam yanlış alarm atağı sayısını "cevap" adıyla sakla.  
> Test: Bir kaydın alarm atakları dakikalarının yarısından bir fazlasını geçemez ve yükler 0 ile 100 arasında olmalı.  

In [ ]:
#@title Adım 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _minutes_table(["kayıt", "başlangıç (s)", "AF", "AF payı", "olasılık"], sets=("test",))
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "cevap", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _minutes_table(["kayıt", "başlangıç (s)", "AF", "AF payı", "olasılık"], sets=("test",))


def ataklar(alarm, af):
    diziler, ilk = [], None
    for k, a in enumerate(list(alarm) + [False]):
        if a and ilk is None:
            ilk = k
        if not a and ilk is not None:
            diziler.append((ilk, k - 1))
            ilk = None
    return len(diziler), sum(1 for s, e in diziler if af[s:e + 1].sum() == 0)


d = veri.sort_values(["kayıt", "başlangıç (s)"])
esik = 0.37
satirlar = []
for kayit, parca in d.groupby("kayıt"):
    alarm = parca["olasılık"].to_numpy() >= esik
    n, yanlis = ataklar(alarm, parca["AF"].to_numpy())
    satirlar.append({"kayıt": kayit, "dakika": len(parca),
                         "AF yükü, işaretleme (%)": round(100 * parca["AF payı"].mean(), 1),
                         "AF yükü, model (%)": round(100 * alarm.mean(), 1),
                         "alarm atakları": n, "yanlış alarm atakları": yanlis})
tablo = pd.DataFrame(satirlar)
print(tablo.to_string(index=False))
cevap = int(tablo["yanlış alarm atakları"].sum())
teach.check_step("Adım 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Test kayıtları yaklaşık 50 saatte 22 yanlış alarm atağı verir. Modelin yükleri, prototipin raporunun ihtiyaç duyduğu gibi, kayıtların çoğunda işaretlemedekileri yakından izler.

**1. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#stop-1) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 2: Açıklama, rapor ve servis

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 5-8. bölümler*

Bu durak bir alarmı ve bir hastayı izler. 08219 kaydındaki bir yanlış alarm dakikası dört sayısının katkılarıyla açıklanır, telemetri servisindeki VC-0128'in sonucu bir HL7 FHIR raporu olarak yazılır ve aynı hastanın vuru zamanları dersin tahmin servisine gönderilir.

**Durağın sonunda:** Yanlış alarm dakikasının AF olasılığını (0,985), VC-0128'in raporundaki AF yükünü (yüzde 15,5) ve servisin cevabındaki AF dakikalarını (222) bulacaksınız.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test"}


def _course():
    """AFDB'nin 25 kaydının ve telemetri servisindeki dört hastanın dakikaları; RR aralıkları, beş sayısı ve 1.0 sürümünün olasılıklarıyla birlikte (bir kez hesaplanır, yaklaşık yirmi saniye sürer)."""
    if "_course_cache" in globals():
        return _course_cache
    SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
    episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
    rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                            for record in afdb.RECORDS for start, end, name in episodes[record]])
    hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
    hours["set"] = hours.index.map(SPLIT)

    WINDOW_S = 60


    def minutes_of(beat_s, episodes=None, record=""):
        """Bir kaydın bir dakikalık pencereleri: Her dakikada biten RR aralıkları ve ritim biliniyorsa dakikanın AF içinde geçen payı."""
        rr = np.diff(beat_s)
        rr_end = beat_s[1:]
        rows = []
        for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
            x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
            if len(x) < 20 or x.max() > 3.0:
                continue
            row = {"record": record, "start_s": float(start), "rr": x}
            if episodes is not None:
                af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
                row["af_share"] = af_s / WINDOW_S
            rows.append(row)
        return rows

    minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
    minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
    minutes["set"] = minutes["record"].map(SPLIT)


    def sample_entropy(rr, m=1, r=0.03):
        """Örneklem entropisi: İlk m aralığı birbirine benzeyen, yani en fazla r saniye farklı olan iki dizinin m + 1 aralıkta da benzer olma olasılığının logaritmasının eksi işaretlisi."""
        n = len(rr)

        def pairs(length):
            t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
            alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
            return (alike.sum() - len(t)) / 2

        b, a = pairs(m), pairs(m + 1)
        return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


    def features(rr):
        """Bir dakikanın beş sayısı: Kalp hızı, değişim katsayısı, ortalama RR'ye bölünmüş RMSSD, pNN50 ve örneklem entropisi."""
        d = np.diff(rr)
        return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
                "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

    minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

    V1_0 = ["hr", "cv", "rmssd", "pnn50"]
    LOGGED = ["cv", "rmssd"]


    def design(table, names):
        """Modelin değişkenleri: CV'nin ve RMSSD'nin logaritması, ötekiler olduğu gibi."""
        X = table[names].astype(float).copy()
        for name in names:
            if name in LOGGED:
                X[name] = np.log(X[name])
        return X


    def fit_model(names, version, training):
        """Standartlaştırılmış değişkenlerle bir lojistik regresyon; JSON'un saklayabileceği düz bir sözlük olarak tutulur."""
        X = design(training, names)
        mean, sd = X.mean(), X.std()
        lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
        return {"name": "RR aralıklarından AF", "version": version, "features": names,
                "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
                "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


    def af_probability(model, table):
        """Bir tablonun her dakikası için AF olasılığı."""
        names = model["features"]
        z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
        w = np.array([model["coef"][name] for name in names])
        return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


    def choose_threshold(p, y, sensitivity=0.95):
        """AF dakikalarının istenen payını hâlâ bulan, 0,01 adımlı en yüksek eşik."""
        grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
        return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


    def alarm_episodes(alarm):
        """Art arda gelen alarm dakikalarının dizileri: (ilk, son) konumları."""
        runs, first = [], None
        for k, a in enumerate(alarm):
            if a and first is None:
                first = k
            if not a and first is not None:
                runs.append((first, k - 1))
                first = None
        if first is not None:
            runs.append((first, len(alarm) - 1))
        return runs


    def by_record(table, p, threshold):
        """Her kayıt için işaretlemenin ve modelin AF yükü, alarm atakları ve bunların yanlış olanları."""
        rows = []
        for record, part in table.assign(p=p).groupby("record", sort=True):
            alarm = (part["p"] >= threshold).to_numpy()
            runs = alarm_episodes(alarm)
            false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
            rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                         "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
        return pd.DataFrame(rows).set_index("record")

    training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
    model = fit_model(V1_0, "1.0.0", training)
    p_val = af_probability(model, validation)
    model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())


    def contributions(model, row):
        """Her değişkenin bir dakikadaki AF log-odds değerine kattığı: Katsayısı çarpı z-skoru."""
        x = design(pd.DataFrame([row]), model["features"]).iloc[0]
        z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
        return {name: model["coef"][name] * z[name] for name in model["features"]}


    def predict(request, model, digest):
        """Tahmin servisi: Bir kaydın vuru zamanlarını içeren istek girer, sonuç çıkar; ikisi de JSON'un taşıyabileceği sözlüklerdir."""
        table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
        table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
        p = af_probability(model, table)
        alarm = p >= model["threshold"]
        runs = alarm_episodes(alarm)
        return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
                "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
                "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

    ward = holter.ward()
    ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                                 for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
    ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
    ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))
    minutes["p10"] = af_probability(model, minutes)
    ward_minutes["p10"] = af_probability(model, ward_minutes)
    digest = hashlib.sha256(json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    globals()["_course_cache"] = dict(minutes=minutes, model=model, digest=digest, ward=ward, ward_minutes=ward_minutes,
                                      contributions=contributions, predict=predict, design=design, by_record=by_record)
    return _course_cache


def _minutes_table(columns, sets=("training", "validation", "test")):
    """Seçilen sütunlarla AFDB dakikalarının bir tablosu."""
    m = _course()["minutes"]
    m = m[m["set"].isin(sets)]
    cols = {"kayıt": m["record"], "küme": m["set"].map(_SETS), "başlangıç (s)": m["start_s"].round(2),
            "AF": m["af"], "AF payı": m["af_share"].round(3), "RR aralıkları (s)": [list(np.round(x, 3)) for x in m["rr"]],
            "kalp hızı": m["hr"].round(2), "CV": m["cv"].round(4), "RMSSD/RR": m["rmssd"].round(4), "pNN50": m["pnn50"].round(4),
            "olasılık": m["p10"].round(4)}
    return pd.DataFrame({k: v for k, v in cols.items() if k in columns}).reset_index(drop=True)



def _start_service(port=8765):
    """Dersin tahmin servisini bu bilgisayarda, verilen kapıda, her çekirdek için bir kez başlatır."""
    if "_service" in globals():
        return _service
    import http.server
    import threading
    c = _course()

    class _Handler(http.server.BaseHTTPRequestHandler):
        def do_POST(self):
            body = self.rfile.read(int(self.headers["Content-Length"]))
            out = json.dumps(c["predict"](json.loads(body), c["model"], c["digest"])).encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(out)

        def log_message(self, *args):
            pass

    try:
        server = http.server.ThreadingHTTPServer(("127.0.0.1", port), _Handler)
    except OSError as error:
        raise RuntimeError("8765 kapısı kullanımda; belki başka bir defter kullanıyordur. O defteri kapatın ya da bu çekirdeği yeniden başlatın ve hücreyi yeniden çalıştırın.") from error
    threading.Thread(target=server.serve_forever, daemon=True).start()
    globals()["_service"] = server
    return server


_c = _course()
print(len(_c["minutes"]), "dakika AFDB'de,", len(_c["ward_minutes"]), "dakika telemetri servisinde")
print("2. durak hazır.")

### Adım 2.1: Dakika neden alarm verdi?

**Amaç.** Bu adımda bir yanlış alarm dakikasında dört sayının her birinin AF'nin log-odds değerine kattığı değer hesaplanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Hekim 08219 kaydındaki bir yanlış alarmı açıyor ve EKG şeridine bakmadan önce dakikanın hangi sayılarının modeli alarm vermeye yönelttiğini soruyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tablo, MIT-BIH Atriyal Fibrilasyon Veri Tabanı'nın 08219 kaydından, AF olmadığı hâlde dersin bir lojistik regresyonunun alarm verdiği bir dakikayı tarif eder. İlk dört satır dakikanın dört sayısıdır: "sayı" sayıyı adlandırır, "dakikadaki değer" değerini verir, "önce logaritma" modelin her şeyden önce değerin doğal logaritmasını alıp almadığını söyler ("evet" ya da "hayır"); "ortalama, eğitim" ve "standart sapma, eğitim" değerin ya da logaritmasının eğitim dakikalarındaki ortalaması ve standart sapmasıdır; "katsayı" ise sayının modeldeki katsayısıdır. Son satır olan "sabit terim", "katsayı" sütununda yalnızca modelin sabit terimini tutar.  
> Görev: Dört sayının her biri için "önce logaritma" evet diyorsa değerin doğal logaritmasını al, ortalamayı çıkar ve standart sapmaya bölerek z-skorunu bul; z-skorunu katsayıyla çarp. Bu, sayının log-odds değerine katkısıdır. Dakikanın log-odds değerini bulmak için sabit terim ile dört katkıyı topla ve 1 / (1 + e üzeri eksi log-odds) ile bir olasılığa dönüştür.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her sayının z-skorunu ve katkısını bir tabloda göster, log-odds değerini ve olasılığı yazdır; olasılığı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Sabit terim ile dört katkının toplamı log-odds değerini vermeli ve olasılık 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_m = _course()["model"]
_x = _course()["minutes"]
_x = _x[(_x["record"] == "08219") & (_x["af"] == 0) & (_x["p10"] >= _m["threshold"])]
_x = _x.iloc[len(_x) // 2]
_names = {"hr": "kalp hızı", "cv": "CV", "rmssd": "RMSSD/RR", "pnn50": "pNN50"}
veri = pd.DataFrame({"sayı": [_names[k] for k in _m["features"]] + ["sabit terim"],
                     "dakikadaki değer": [round(float(_x[k]), 4) for k in _m["features"]] + [np.nan],
                     "önce logaritma": ["evet" if k in ("cv", "rmssd") else "hayır" for k in _m["features"]] + [""],
                     "ortalama, eğitim": [_m["mean"][k] for k in _m["features"]] + [np.nan],
                     "standart sapma, eğitim": [_m["sd"][k] for k in _m["features"]] + [np.nan],
                     "katsayı": [_m["coef"][k] for k in _m["features"]] + [_m["intercept"]]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "cevap", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_m = _course()["model"]
_x = _course()["minutes"]
_x = _x[(_x["record"] == "08219") & (_x["af"] == 0) & (_x["p10"] >= _m["threshold"])]
_x = _x.iloc[len(_x) // 2]
_names = {"hr": "kalp hızı", "cv": "CV", "rmssd": "RMSSD/RR", "pnn50": "pNN50"}
veri = pd.DataFrame({"sayı": [_names[k] for k in _m["features"]] + ["sabit terim"],
                     "dakikadaki değer": [round(float(_x[k]), 4) for k in _m["features"]] + [np.nan],
                     "önce logaritma": ["evet" if k in ("cv", "rmssd") else "hayır" for k in _m["features"]] + [""],
                     "ortalama, eğitim": [_m["mean"][k] for k in _m["features"]] + [np.nan],
                     "standart sapma, eğitim": [_m["sd"][k] for k in _m["features"]] + [np.nan],
                     "katsayı": [_m["coef"][k] for k in _m["features"]] + [_m["intercept"]]})
d = veri
satirlar = d.iloc[:4]
x = np.where(satirlar["önce logaritma"] == "evet", np.log(satirlar["dakikadaki değer"]),
               satirlar["dakikadaki değer"])
z = (x - satirlar["ortalama, eğitim"]) / satirlar["standart sapma, eğitim"]
katki = satirlar["katsayı"] * z
eta = float(d.iloc[4]["katsayı"] + katki.sum())
print(pd.DataFrame({"sayı": satirlar["sayı"], "z": z.round(3), "katkı": katki.round(3)}).to_string(index=False))
print("log-odds", round(eta, 3), "| olasılık", round(1 / (1 + np.exp(-eta)), 3))
cevap = round(float(1 / (1 + np.exp(-eta))), 3)
teach.check_step("Adım 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** Dakika 0,985 olasılık alır ve bunun büyük kısmı pNN50'den gelir: Ardışık aralıkları birbirinden çok farklıdır. Katkılar modelin neden alarm verdiğini söyler; aralıkların neden farklı olduğunu ise EKG şeridi gösterir: AF değil, erken vurular.

### Adım 2.2: HL7 FHIR kaynakları olarak rapor

**Amaç.** Bu adımda VC-0128'in sonucu, bir hastane bilgi sisteminin okuyabileceği bir HL7 FHIR R4 raporu olarak yazılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Prototipin 24 saatlik raporu, bir kardiyolog onaylayana ya da reddedene kadar ön rapor olarak hastanın kaydına ulaşmalı ve raporu hangi modelin yazdığını belirtmelidir.  
> Veri: Defterde "veri" adlı ve tek satırlık bir tablo vardır. Satır, dersin bir karar destek prototipinin sanal kliniğin telemetri servisindeki bir hasta olan VC-0128 için verdiği sonuçtur; veriler sentetiktir. "hasta" hastanın kimliğini, "AF yükü (%)" incelenen dakikalardan atriyal fibrilasyon (AF) alarmı verenlerin payını, "alarm atakları" art arda gelen alarm dakikalarından oluşan dizilerin sayısını, "en uzun atak (dakika)" ise bunların en uzununu verir. "kaydın başlangıcı" ve "kaydın sonu" 24 saatlik kaydın dönemini; "model", "sürüm" ve "SHA-256" modeli ve dosyasının parmak izini; "raporun zamanı" ise raporun yazıldığı zamanı verir. Bütün zamanlar UTC'ye göre farklarıyla birlikte ISO biçimindedir.  
> Görev: "collection" türünde, dört kaynaktan oluşan bir FHIR Bundle yaz; her kaynağın bir kimliği ve http://example.org/fhir altında bir fullUrl değeri olsun. Modelin adını, sürümünü ve SHA-256 değerini içeren bir not taşıyan "af-rr" kimlikli bir Device. Durumu "preliminary", kategorisi "procedure" olan; http://example.org/vclinic sisteminden "af-burden" yerel kodunu "AF yükü" metniyle taşıyan; hastayı konu olarak, kaydın dönemini, AF yükünü UCUM'un "%" biriminde bir nicelik olarak (sistem http://unitsofmeasure.org, kod "%") ve Device'a bir başvuruyu içeren "af-burden" kimlikli bir Observation. Durumu "preliminary" olan; "Holter kaydı, AF analizi" metnini, hastayı, raporun zamanını, Observation'a bir başvuruyu ve kelimelerle bir sonucu içeren "af-report" kimlikli bir DiagnosticReport. Raporu hedef olarak adlandıran, raporun zamanını kaydeden ve Device'ı yazar olarak veren bir Provenance. Ardından hasta dışında paketin içindeki her başvurunun paketteki bir kaynağa ulaştığını denetle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Observation'ı JSON olarak yazdır, kaynakların ve bir kaynağa ulaşmayan başvuruların sayısını yazdır; Observation'daki AF yükünün değerini "cevap" adıyla sakla.  
> Test: Paket dört kaynak içermeli, içindeki hiçbir başvuru eksik bir kaynağı göstermemeli ve iki durum da "preliminary" olmalı.  

In [ ]:
#@title Adım 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
_w = _c["ward_minutes"]
_p = _w[_w["record"] == "VC-0128"]["p10"].to_numpy() >= _c["model"]["threshold"]
_eps, _start = [], None
for _k, _a in enumerate(list(_p) + [False]):
    if _a and _start is None:
        _start = _k
    if not _a and _start is not None:
        _eps.append(_k - _start)
        _start = None
veri = pd.DataFrame([{"hasta": "VC-0128", "AF yükü (%)": round(100 * float(_p.mean()), 1), "alarm atakları": len(_eps),
                      "en uzun atak (dakika)": max(_eps), "kaydın başlangıcı": "2026-11-18T08:00:00+03:00",
                      "kaydın sonu": "2026-11-19T08:00:00+03:00", "model": _c["model"]["name"], "sürüm": _c["model"]["version"],
                      "SHA-256": _c["digest"], "raporun zamanı": "2026-11-19T09:30:00+03:00"}])
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "cevap", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
_w = _c["ward_minutes"]
_p = _w[_w["record"] == "VC-0128"]["p10"].to_numpy() >= _c["model"]["threshold"]
_eps, _start = [], None
for _k, _a in enumerate(list(_p) + [False]):
    if _a and _start is None:
        _start = _k
    if not _a and _start is not None:
        _eps.append(_k - _start)
        _start = None
veri = pd.DataFrame([{"hasta": "VC-0128", "AF yükü (%)": round(100 * float(_p.mean()), 1), "alarm atakları": len(_eps),
                      "en uzun atak (dakika)": max(_eps), "kaydın başlangıcı": "2026-11-18T08:00:00+03:00",
                      "kaydın sonu": "2026-11-19T08:00:00+03:00", "model": _c["model"]["name"], "sürüm": _c["model"]["version"],
                      "SHA-256": _c["digest"], "raporun zamanı": "2026-11-19T09:30:00+03:00"}])
r = veri.iloc[0]
durum = "preliminary"
cihaz = {"resourceType": "Device", "id": "af-rr", "deviceName": [{"name": r["model"], "type": "model-name"}],
                "version": [{"value": r["sürüm"]}], "note": [{"text": "SHA-256 " + r["SHA-256"]}]}
gozlem = {"resourceType": "Observation", "id": "af-burden", "status": durum,
                      "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "procedure"}]}],
                      "code": {"coding": [{"system": "http://example.org/vclinic", "code": "af-burden"}], "text": "AF yükü"},
                      "subject": {"reference": "Patient/" + r["hasta"]},
                      "effectivePeriod": {"start": r["kaydın başlangıcı"], "end": r["kaydın sonu"]},
                      "valueQuantity": {"value": float(r["AF yükü (%)"]), "unit": "%", "system": "http://unitsofmeasure.org", "code": "%"},
                      "device": {"reference": "Device/af-rr"}}
rapor = {"resourceType": "DiagnosticReport", "id": "af-report", "status": durum, "code": {"text": "Holter kaydı, AF analizi"},
                "subject": {"reference": "Patient/" + r["hasta"]}, "issued": r["raporun zamanı"],
                "result": [{"reference": "Observation/af-burden"}],
                "conclusion": f"AF yükü {r['AF yükü (%)']} %, {r['alarm atakları']} alarm atağı; en uzunu {r['en uzun atak (dakika)']} dakika."}
kaynak = {"resourceType": "Provenance", "id": "af-provenance", "target": [{"reference": "DiagnosticReport/af-report"}],
                     "recorded": r["raporun zamanı"], "agent": [{"type": {"text": "author"}, "who": {"reference": "Device/af-rr"}}]}
paket = {"resourceType": "Bundle", "type": "collection",
                "entry": [{"fullUrl": f"http://example.org/fhir/{x['resourceType']}/{x['id']}", "resource": x}
                          for x in (cihaz, gozlem, rapor, kaynak)]}
mevcut = {f"{e['resource']['resourceType']}/{e['resource']['id']}" for e in paket["entry"]}
basvurular = re.findall(r'"reference": "([^"]+)"', json.dumps(paket))
acik = [x for x in basvurular if x.split("/")[0] not in ("Patient", "Practitioner") and x not in mevcut]
print(json.dumps(gozlem, indent=2, ensure_ascii=False))
print(len(paket["entry"]), "kaynak;", len(basvurular), "başvuru; bir kaynağa ulaşmayan:", len(acik))
cevap = float(gozlem["valueQuantity"]["value"])
teach.check_step("Adım 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** Rapor AF yükünü yüzde 15,5 olarak, ön rapor durumunda verir. Kardiyoloğun kararından sonra durum kesinleşir ve rapor kardiyoloğun adını taşır; etkileşimli labın B bölümü her değişikliği gösterir.

### Adım 2.3: Tahmin servisine bir istek

**Amaç.** Bu adımda VC-0128'in vuru zamanları dersin tahmin servisine gönderilir ve servisin cevabı okunur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Servisin web arayüzü modeli kendisi tutmamalı: Sonucu ve modelin sürümünü veren tahmin servisine sormalıdır.  
> Veri: Defterde "veri" adlı ve tek sütunlu bir tablo vardır: "vuru zamanı (s)". Sütun, sanal kliniğin telemetri servisindeki bir hasta olan VC-0128'in 24 saatlik kaydında monitörün bulduğu 104.314 vurunun zamanlarını, kaydın başından itibaren saniye cinsinden verir; veriler sentetiktir. Veri hücresi ayrıca dersin bir tahmin servisini bu bilgisayarda http://127.0.0.1:8765/predict adresinde başlatmıştır. Servis "patient_id" ve "beat_times_s" (saniye cinsinden vuru zamanlarının listesi) alanlarını içeren JSON biçiminde bir istek alır; modeli, sürümünü ve parmak izini, eşiği, incelenen dakikaları, atriyal fibrilasyon (AF) alarmı veren dakikaları, yüzde cinsinden AF yükünü ve alarm ataklarını içeren JSON biçiminde bir cevap verir.  
> Görev: "VC-0128" hastası için bütün vuru zamanlarını içeren isteği hazırla, POST yöntemiyle JSON olarak servise gönder ve cevabı JSON olarak oku. Servis bu bilgisayarda çalıştığı için istek bir vekil sunucudan (proxy) geçmemelidir.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İsteğin kilobayt cinsinden büyüklüğünü, modeli, sürümünü ve parmak izinin ilk on iki basamağını, dakikaları, AF dakikalarını ve AF yükünü yazdır; alarm ataklarını bir tablo olarak göster ve AF dakikalarının sayısını "cevap" adıyla sakla.  
> Test: Cevaptaki AF yükü, AF dakikalarının dakikalara bölümünün 100 katına bir ondalık basamakla yuvarlanmış olarak eşit olmalı.  

In [ ]:
#@title Adım 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_start_service()
veri = pd.DataFrame({"vuru zamanı (s)": np.round(_course()["ward"]["VC-0128"]["detected_s"], 3)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "cevap", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_start_service()
veri = pd.DataFrame({"vuru zamanı (s)": np.round(_course()["ward"]["VC-0128"]["detected_s"], 3)})
istek = {"patient_id": "VC-0128", "beat_times_s": [float(t) for t in veri["vuru zamanı (s)"]]}
govde = json.dumps(istek).encode("utf-8")
gonder = urllib.request.Request("http://127.0.0.1:8765/predict", data=govde, headers={"Content-Type": "application/json"})
yerel = urllib.request.build_opener(urllib.request.ProxyHandler({}))
yanit = json.loads(yerel.open(gonder, timeout=120).read().decode("utf-8"))
print(round(len(govde) / 1024), "kB gönderildi ·", yanit["model"], yanit["version"], yanit["sha256"][:12])
print("dakika", yanit["minutes"], "| AF dakikası", yanit["af_minutes"], "| AF yükü", yanit["burden_percent"], "%")
print(pd.DataFrame(yanit["episodes"]))
cevap = int(yanit["af_minutes"])
teach.check_step("Adım 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** Servis 222 AF dakikası bulur ve cevabında modelini ve sürümünü adlandırır. Onu çağıran bir program modelin bir kopyasına ihtiyaç duymaz; yeni bir sürüm de servis tarafında tek bir dosyayı değiştirir.

**2. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#stop-2) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 3: Kullanıcı testi ve 1.1 sürümü

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 9-10. bölümler*

Bu durak kullanıcı testini yapar ve prototipi iyileştirir. Telemetri servisindeki dört hastanın alarm atakları bir hekimin 24 saatte neyi inceleyeceğini gösterir; 1.1 sürümü örneklem entropisini ekler ve eğitim kayıtlarında yeniden öğrenir; test kayıtları da iki sürümü bir kez değerlendirir.

**Durağın sonunda:** Serviste 24 saatteki yanlış alarm ataklarını (76), 1.1 sürümünün doğrulama kayıtlarındaki yanlış alarm ataklarını (69) ve 1.1 sürümünün test kayıtlarındaki duyarlılığını (0,924) bulacaksınız.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test"}


def _course():
    """AFDB'nin 25 kaydının ve telemetri servisindeki dört hastanın dakikaları; RR aralıkları, beş sayısı ve 1.0 sürümünün olasılıklarıyla birlikte (bir kez hesaplanır, yaklaşık yirmi saniye sürer)."""
    if "_course_cache" in globals():
        return _course_cache
    SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
    episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
    rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                            for record in afdb.RECORDS for start, end, name in episodes[record]])
    hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
    hours["set"] = hours.index.map(SPLIT)

    WINDOW_S = 60


    def minutes_of(beat_s, episodes=None, record=""):
        """Bir kaydın bir dakikalık pencereleri: Her dakikada biten RR aralıkları ve ritim biliniyorsa dakikanın AF içinde geçen payı."""
        rr = np.diff(beat_s)
        rr_end = beat_s[1:]
        rows = []
        for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
            x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
            if len(x) < 20 or x.max() > 3.0:
                continue
            row = {"record": record, "start_s": float(start), "rr": x}
            if episodes is not None:
                af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
                row["af_share"] = af_s / WINDOW_S
            rows.append(row)
        return rows

    minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
    minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
    minutes["set"] = minutes["record"].map(SPLIT)


    def sample_entropy(rr, m=1, r=0.03):
        """Örneklem entropisi: İlk m aralığı birbirine benzeyen, yani en fazla r saniye farklı olan iki dizinin m + 1 aralıkta da benzer olma olasılığının logaritmasının eksi işaretlisi."""
        n = len(rr)

        def pairs(length):
            t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
            alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
            return (alike.sum() - len(t)) / 2

        b, a = pairs(m), pairs(m + 1)
        return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


    def features(rr):
        """Bir dakikanın beş sayısı: Kalp hızı, değişim katsayısı, ortalama RR'ye bölünmüş RMSSD, pNN50 ve örneklem entropisi."""
        d = np.diff(rr)
        return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
                "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

    minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

    V1_0 = ["hr", "cv", "rmssd", "pnn50"]
    LOGGED = ["cv", "rmssd"]


    def design(table, names):
        """Modelin değişkenleri: CV'nin ve RMSSD'nin logaritması, ötekiler olduğu gibi."""
        X = table[names].astype(float).copy()
        for name in names:
            if name in LOGGED:
                X[name] = np.log(X[name])
        return X


    def fit_model(names, version, training):
        """Standartlaştırılmış değişkenlerle bir lojistik regresyon; JSON'un saklayabileceği düz bir sözlük olarak tutulur."""
        X = design(training, names)
        mean, sd = X.mean(), X.std()
        lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
        return {"name": "RR aralıklarından AF", "version": version, "features": names,
                "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
                "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


    def af_probability(model, table):
        """Bir tablonun her dakikası için AF olasılığı."""
        names = model["features"]
        z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
        w = np.array([model["coef"][name] for name in names])
        return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


    def choose_threshold(p, y, sensitivity=0.95):
        """AF dakikalarının istenen payını hâlâ bulan, 0,01 adımlı en yüksek eşik."""
        grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
        return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


    def alarm_episodes(alarm):
        """Art arda gelen alarm dakikalarının dizileri: (ilk, son) konumları."""
        runs, first = [], None
        for k, a in enumerate(alarm):
            if a and first is None:
                first = k
            if not a and first is not None:
                runs.append((first, k - 1))
                first = None
        if first is not None:
            runs.append((first, len(alarm) - 1))
        return runs


    def by_record(table, p, threshold):
        """Her kayıt için işaretlemenin ve modelin AF yükü, alarm atakları ve bunların yanlış olanları."""
        rows = []
        for record, part in table.assign(p=p).groupby("record", sort=True):
            alarm = (part["p"] >= threshold).to_numpy()
            runs = alarm_episodes(alarm)
            false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
            rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                         "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
        return pd.DataFrame(rows).set_index("record")

    training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
    model = fit_model(V1_0, "1.0.0", training)
    p_val = af_probability(model, validation)
    model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())


    def contributions(model, row):
        """Her değişkenin bir dakikadaki AF log-odds değerine kattığı: Katsayısı çarpı z-skoru."""
        x = design(pd.DataFrame([row]), model["features"]).iloc[0]
        z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
        return {name: model["coef"][name] * z[name] for name in model["features"]}


    def predict(request, model, digest):
        """Tahmin servisi: Bir kaydın vuru zamanlarını içeren istek girer, sonuç çıkar; ikisi de JSON'un taşıyabileceği sözlüklerdir."""
        table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
        table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
        p = af_probability(model, table)
        alarm = p >= model["threshold"]
        runs = alarm_episodes(alarm)
        return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
                "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
                "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

    ward = holter.ward()
    ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                                 for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
    ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
    ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))
    minutes["p10"] = af_probability(model, minutes)
    ward_minutes["p10"] = af_probability(model, ward_minutes)
    digest = hashlib.sha256(json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    globals()["_course_cache"] = dict(minutes=minutes, model=model, digest=digest, ward=ward, ward_minutes=ward_minutes,
                                      contributions=contributions, predict=predict, design=design, by_record=by_record)
    return _course_cache


def _minutes_table(columns, sets=("training", "validation", "test")):
    """Seçilen sütunlarla AFDB dakikalarının bir tablosu."""
    m = _course()["minutes"]
    m = m[m["set"].isin(sets)]
    cols = {"kayıt": m["record"], "küme": m["set"].map(_SETS), "başlangıç (s)": m["start_s"].round(2),
            "AF": m["af"], "AF payı": m["af_share"].round(3), "RR aralıkları (s)": [list(np.round(x, 3)) for x in m["rr"]],
            "kalp hızı": m["hr"].round(2), "CV": m["cv"].round(4), "RMSSD/RR": m["rmssd"].round(4), "pNN50": m["pnn50"].round(4),
            "olasılık": m["p10"].round(4)}
    return pd.DataFrame({k: v for k, v in cols.items() if k in columns}).reset_index(drop=True)


_c = _course()
print(len(_c["minutes"]), "dakika AFDB'nin 25 kaydında")
print("3. durak hazır.")

### Adım 3.1: Servisin alarm atakları

**Amaç.** Bu adımda servis hekiminin 24 saatte inceleyeceği alarm atakları ve bunların yanlış olanları sayılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kullanıcı testi, prototipin servis hekimine ne kadar iş çıkardığını ve yanlış alarmlarının nereden geldiğini soruyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda sanal kliniğin telemetri servisindeki dört hastanın 24 saatlik kayıtlarının dakikaları bulunur; veriler sentetiktir: VC-0128 (paroksismal atriyal fibrilasyon, atriyal erken vurular ve iki elektrot gürültüsü dönemi), VC-0347 (sık atriyal erken vurulu sinüs ritmi, AF yok), VC-0519 (kalıcı AF) ve VC-0733 (uzun elektrot gürültüsü dönemleri olan sinüs ritmi, AF yok). Her satır bir dakikadır: "hasta" hastayı, "başlangıç (sa)" dakikanın kaydın başından itibaren saat cinsinden başlangıcını verir; dakikanın yarısından fazlası kaydın gerçeğinde AF içindeyse "AF" 1 olur; "olasılık" ise dersin bir modelinin 1.0 sürümünün verdiği AF olasılığıdır. Model, olasılık en az 0,37 olduğunda alarm verir.  
> Görev: Her hasta içinde dakikaları başlangıçlarına göre sırala. Bir alarm atağı, alarm veren art arda dakikalardan oluşan bir dizidir; içinde AF olan hiçbir dakika bulunmayan atak ise bir yanlış alarm atağıdır. Her hasta için alarm dakikalarını, yanlış alarm dakikalarını (AF olmayan alarm dakikaları), alarm ataklarını ve yanlış alarm ataklarını say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her hasta için bir satırdan oluşan bir tablo göster, alarm ataklarının ve yanlış olanların toplamını yazdır; toplam yanlış alarm atağı sayısını "cevap" adıyla sakla.  
> Test: Hiç AF dakikası olmayan bir hastanın doğru alarm atağı olamaz ve yanlış alarm atakları alarm ataklarından fazla olamaz.  

In [ ]:
#@title Adım 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_w = _course()["ward_minutes"]
veri = pd.DataFrame({"hasta": _w["record"], "başlangıç (sa)": (_w["start_s"] / 3600).round(4), "AF": _w["af"],
                     "olasılık": _w["p10"].round(4)}).reset_index(drop=True)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
teach.check_step("Adım 3.1", "cevap", globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_w = _course()["ward_minutes"]
veri = pd.DataFrame({"hasta": _w["record"], "başlangıç (sa)": (_w["start_s"] / 3600).round(4), "AF": _w["af"],
                     "olasılık": _w["p10"].round(4)}).reset_index(drop=True)


def ataklar(alarm, af):
    diziler, ilk = [], None
    for k, a in enumerate(list(alarm) + [False]):
        if a and ilk is None:
            ilk = k
        if not a and ilk is not None:
            diziler.append((ilk, k - 1))
            ilk = None
    return len(diziler), sum(1 for s, e in diziler if af[s:e + 1].sum() == 0)


d = veri.sort_values(["hasta", "başlangıç (sa)"])
esik = 0.37
satirlar = []
for hasta, parca in d.groupby("hasta"):
    alarm = parca["olasılık"].to_numpy() >= esik
    n, yanlis = ataklar(alarm, parca["AF"].to_numpy())
    satirlar.append({"hasta": hasta, "alarm dakikası": int(alarm.sum()),
                         "yanlış alarm dakikası": int((alarm & (parca["AF"].to_numpy() == 0)).sum()),
                         "alarm atakları": n, "yanlış alarm atakları": yanlis})
tablo = pd.DataFrame(satirlar)
print(tablo.to_string(index=False))
print("24 saatte incelenecek:", int(tablo["alarm atakları"].sum()), "atak; yanlış olan:",
      int(tablo["yanlış alarm atakları"].sum()))
cevap = int(tablo["yanlış alarm atakları"].sum())
teach.check_step("Adım 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** Hekim 24 saatte 76 yanlış alarm atağını incelemek zorunda kalır. Bunların neredeyse hepsi, atriyal erken vuruları dört sayıya düzensiz görünen VC-0347'de ve kayıt cihazının gürültüde vuruları kaçırıp eklediği VC-0733'tedir.

### Adım 3.2: Örneklem entropisiyle 1.1 sürümü

**Amaç.** Bu adımda örneklem entropisi beşinci sayı olarak eklenir, 1.1 sürümü eğitim kayıtlarında kurulur ve doğrulama kayıtlarındaki yanlış alarm atakları sayılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kullanıcı testi, yanlış alarmların çoğunu erken vuruların verdiğini göstermiştir. Erken vurular aynı kısa ve uzun aralıkları tekrarlar, AF'de ise örüntü yoktur; bu yüzden örüntüyü ölçen bir sayı yardımcı olabilir. Test kayıtlarına dokunulmamalıdır.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda MIT-BIH Atriyal Fibrilasyon Veri Tabanı'ndaki (AFDB) çoğu paroksismal atriyal fibrilasyonu (AF) olan kişilerin yaklaşık on saatlik 25 Holter kaydından elde edilen bir dakikalık pencereler arasından eğitim ve doğrulama dakikaları bulunur; test kayıtları tabloda yoktur. Her satır bir dakikadır: "kayıt" kaydı, "küme" kaydın kümesini ("eğitim", 15 kayıt; "doğrulama", 5 kayıt), "başlangıç (s)" ise dakikanın saniye cinsinden başlangıcını verir. Dakikanın yarısından fazlası uzmanların işaretlediği AF içindeyse "AF" 1 olur. "RR aralıkları (s)" sütununun her hücresi dakikanın RR aralıklarını saniye cinsinden ve sırasıyla tutar. Dakikanın dört sayısı, önceki adımlarda olduğu gibi "kalp hızı" (dakikada), "CV", "RMSSD/RR" ve "pNN50" sütunlarındadır.  
> Görev: Her dakika için ayrıca RR aralıklarının örneklem entropisini bir aralıklı dizilerle ve 0,03 saniyelik bir toleransla hesapla: Dakikanın ilk n − 1 aralığı arasında en fazla 0,03 saniye farklı olan çiftleri say (B); bu çiftlerden kaçının, her aralığa bir sonraki aralık eklendiğinde de benzer kaldığını say (A). Örneklem entropisi, B'nin A'ya bölümünün doğal logaritmasıdır; A 0 ise 1 say, B 0 ise dakikaya n'nin doğal logaritmasını ver. Eğitim dakikalarında en fazla 3.000 yinelemeyle AF için beş girdili bir lojistik regresyon kur: Kalp hızı, CV'nin doğal logaritması, RMSSD/RR'nin doğal logaritması, pNN50 ve örneklem entropisi; her biri eğitim dakikalarının ortalaması ve standart sapmasıyla bir z-skoruna dönüştürülür. Eşiği yalnızca doğrulama dakikalarında seç: 0,05, 0,06, 0,07 ve bu biçimde 0,95'e kadar giden değerler arasından, AF olan doğrulama dakikalarının en az yüzde 95'inin olasılığının ona eşit ya da ondan büyük olduğu en yüksek değeri al. Ardından her doğrulama kaydı içinde dakikaları başlangıçlarına göre sırala, alarm ataklarını, yani alarm veren art arda dakikaları bul ve içinde AF olan hiçbir dakika bulunmayan yanlış alarm ataklarını say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Doğrulama AUC değerini, eşiği, yanlış alarm ataklarının sayısını ve AF olan ve olmayan eğitim dakikalarının ortanca örneklem entropisini yazdır; yanlış alarm ataklarının sayısını "cevap" adıyla sakla.  
> Test: Hiçbir test dakikası kullanılmamalı ve her dakikanın örneklem entropisi 0 ya da daha büyük olmalı.  

In [ ]:
#@title Adım 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _minutes_table(["kayıt", "küme", "başlangıç (s)", "AF", "RR aralıkları (s)",
                       "kalp hızı", "CV", "RMSSD/RR", "pNN50"], sets=("training", "validation"))
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
teach.check_step("Adım 3.2", "cevap", globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _minutes_table(["kayıt", "küme", "başlangıç (s)", "AF", "RR aralıkları (s)",
                       "kalp hızı", "CV", "RMSSD/RR", "pNN50"], sets=("training", "validation"))


def ataklar(alarm, af):
    diziler, ilk = [], None
    for k, a in enumerate(list(alarm) + [False]):
        if a and ilk is None:
            ilk = k
        if not a and ilk is not None:
            diziler.append((ilk, k - 1))
            ilk = None
    return len(diziler), sum(1 for s, e in diziler if af[s:e + 1].sum() == 0)



def ornek_entropi(rr, r=0.03):
    rr = np.asarray(rr, dtype=float)
    n = len(rr)
    a1 = rr[: n - 1]
    benzer = np.abs(a1[:, None] - a1[None, :]) <= r
    sonraki = np.abs(rr[1:][:, None] - rr[1:][None, :]) <= r
    B = (benzer.sum() - len(a1)) / 2
    A = ((benzer & sonraki).sum() - len(a1)) / 2
    return float(np.log(B / max(A, 1))) if B > 0 else float(np.log(n))


d = veri.copy()
d["örneklem entropisi"] = d["RR aralıkları (s)"].apply(ornek_entropi)
X = pd.DataFrame({"kalp hızı": d["kalp hızı"], "log CV": np.log(d["CV"]),
                    "log RMSSD/RR": np.log(d["RMSSD/RR"]), "pNN50": d["pNN50"], "SampEn": d["örneklem entropisi"]})
eg_ = (d["küme"] == "eğitim").to_numpy()
do_ = ~eg_
ort, ss = X[eg_].mean(), X[eg_].std()
lr = LogisticRegression(max_iter=3000).fit((X[eg_] - ort) / ss, d["AF"][eg_])
p = lr.predict_proba((X[do_] - ort) / ss)[:, 1]
y = d["AF"].to_numpy()[do_]
esik = max(t for t in np.round(np.arange(0.05, 0.96, 0.01), 2) if (p[y == 1] >= t).mean() >= 0.95)
v = d[do_].assign(p_=p).sort_values(["kayıt", "başlangıç (s)"])
yanlis_toplam = 0
for kayit, parca in v.groupby("kayıt"):
    yanlis_toplam += ataklar(parca["p_"].to_numpy() >= esik, parca["AF"].to_numpy())[1]
print("doğrulama AUC", round(roc_auc_score(y, p), 3), "| eşik", esik,
      "| yanlış alarm atakları", yanlis_toplam)
print("ortanca örneklem entropisi, eğitim: AF", round(d[eg_ & (d["AF"] == 1)]["örneklem entropisi"].median(), 2),
      "| öteki dakikalar", round(d[eg_ & (d["AF"] == 0)]["örneklem entropisi"].median(), 2))
cevap = int(yanlis_toplam)
teach.check_step("Adım 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** 1.1 sürümü doğrulama kayıtlarında 69 yanlış alarm atağı verir; 1.0 sürümünde bu sayı 99 olarak bulunmuştu. AF dakikalarının ortanca örneklem entropisi öteki dakikalarınkinin çok üzerindedir.

### Adım 3.3: Test kayıtlarında iki sürüm

**Amaç.** Bu adımda test kayıtları bir kez açılır ve modelin iki sürümü karşılaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Klinik servisteki deneme için bir sürüm seçmeden önce kardiyolog, iki sürümün de görmediği kayıtlarda, 1.1 sürümünün kazanç ve bedelleriyle birlikte değerlendirilmesini istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda MIT-BIH Atriyal Fibrilasyon Veri Tabanı'nın 5 test kaydının, yani çoğu paroksismal atriyal fibrilasyonu (AF) olan kişilerin Holter kayıtlarının bir dakikalık pencereleri bulunur. Her satır bir dakikadır: "kayıt" kaydı, "başlangıç (s)" ise dakikanın saniye cinsinden başlangıcını verir. Dakikanın yarısından fazlası uzmanların işaretlediği AF içindeyse "AF" 1 olur; "AF payı" dakikanın AF içinde geçen payıdır ve 0 ile 1 arasında değer alır. "olasılık, 1.0 sürümü" ve "olasılık, 1.1 sürümü" dersin bir modelinin iki sürümünün verdiği AF olasılıklarıdır. 1.0 sürümü olasılığı en az 0,37, 1.1 sürümü ise olasılığı en az 0,28 olduğunda alarm verir.  
> Görev: Her kayıt içinde dakikaları başlangıçlarına göre sırala. Bir alarm atağı, alarm veren art arda dakikalardan oluşan bir dizidir; içinde AF olan hiçbir dakika bulunmayan atak ise bir yanlış alarm atağıdır. Her sürüm için bütün test dakikalarında AUC değerini, duyarlılığı (alarm veren AF dakikalarının payı) ve özgüllüğü (alarm vermeyen öteki dakikaların payı) hesapla. Test kayıtlarındaki yanlış alarm ataklarını say ve AF yükündeki hatayı hesapla: Her kayıt için alarm veren dakikalarının payının 100 katı ile ortalama AF payının 100 katı arasındaki mutlak fark, kayıtlar üzerinden ortalaması alınarak.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her sürüm için bir satırdan oluşan bir tablo göster; 1.1 sürümünün duyarlılığını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her AUC, duyarlılık ve özgüllük 0 ile 1 arasında olmalı ve her eşik doğrulama kayıtlarında seçilen eşik olmalı.  

In [ ]:
#@title Adım 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
_m = _c["minutes"]
_tr, _te = _m[_m["set"] == "training"], _m[_m["set"] == "test"]
_V11 = _c["model"]["features"] + ["sampen"]
_X = _c["design"](_tr, _V11)
_mu, _sd = _X.mean(), _X.std()
_lr = LogisticRegression(max_iter=3000).fit((_X - _mu) / _sd, _tr["af"])
_z = (_c["design"](_te, _V11) - _mu) / _sd
veri = pd.DataFrame({"kayıt": _te["record"], "başlangıç (s)": _te["start_s"].round(2), "AF": _te["af"],
                     "AF payı": _te["af_share"].round(3), "olasılık, 1.0 sürümü": _te["p10"].round(4),
                     "olasılık, 1.1 sürümü": _lr.predict_proba(_z)[:, 1].round(4)}).reset_index(drop=True)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
teach.check_step("Adım 3.3", "cevap", globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
_m = _c["minutes"]
_tr, _te = _m[_m["set"] == "training"], _m[_m["set"] == "test"]
_V11 = _c["model"]["features"] + ["sampen"]
_X = _c["design"](_tr, _V11)
_mu, _sd = _X.mean(), _X.std()
_lr = LogisticRegression(max_iter=3000).fit((_X - _mu) / _sd, _tr["af"])
_z = (_c["design"](_te, _V11) - _mu) / _sd
veri = pd.DataFrame({"kayıt": _te["record"], "başlangıç (s)": _te["start_s"].round(2), "AF": _te["af"],
                     "AF payı": _te["af_share"].round(3), "olasılık, 1.0 sürümü": _te["p10"].round(4),
                     "olasılık, 1.1 sürümü": _lr.predict_proba(_z)[:, 1].round(4)}).reset_index(drop=True)


def ataklar(alarm, af):
    diziler, ilk = [], None
    for k, a in enumerate(list(alarm) + [False]):
        if a and ilk is None:
            ilk = k
        if not a and ilk is not None:
            diziler.append((ilk, k - 1))
            ilk = None
    return len(diziler), sum(1 for s, e in diziler if af[s:e + 1].sum() == 0)


d = veri.sort_values(["kayıt", "başlangıç (s)"])
satirlar = []
for ad, sutun, esik in (("1.0", "olasılık, 1.0 sürümü", 0.37),
                                              ("1.1", "olasılık, 1.1 sürümü", 0.28)):
    p, y = d[sutun].to_numpy(), d["AF"].to_numpy()
    alarm = p >= esik
    yanlis, hatalar = 0, []
    for kayit, parca in d.groupby("kayıt"):
        a = parca[sutun].to_numpy() >= esik
        yanlis += ataklar(a, parca["AF"].to_numpy())[1]
        hatalar.append(abs(100 * a.mean() - 100 * parca["AF payı"].mean()))
    satirlar.append({"sürüm": ad, "AUC": round(roc_auc_score(y, p), 3),
                         "duyarlılık": round((alarm & (y == 1)).sum() / (y == 1).sum(), 3),
                         "özgüllük": round((~alarm & (y == 0)).sum() / (y == 0).sum(), 3),
                         "yanlış alarm atakları": yanlis, "AF yükündeki hata (puan)": round(np.mean(hatalar), 1)})
tablo = pd.DataFrame(satirlar)
print(tablo.to_string(index=False))
cevap = float(tablo.iloc[1]["duyarlılık"])
teach.check_step("Adım 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** 1.1 sürümü test kayıtlarında daha az yanlış alarm atağı verir, ancak duyarlılığı 0,924 düzeyine düşer. 1.1 sürümünün değişiklik kaydı ikisini de bildirmelidir.

**3. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#stop-3) dönün ve duraktan sonrasını okumaya devam edin.

## Prompt yolunda haftalık proje

*Haftanın projesi: [10. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-10/README.md#haftal%C4%B1k-proje)*

Haftalık proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla da yapılabilir. Önce hazırlık hücresini bir kez çalıştırın; hücre projenin verilerini yükler. Ardından her adım, duraklardaki adımlar gibi ilerler. Tek fark şudur: Projede kontrol hücresi ve hazır kod yoktur, çünkü sonuçlar rapora girer. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler)
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test"}


def _course():
    """AFDB'nin 25 kaydının ve telemetri servisindeki dört hastanın dakikaları; RR aralıkları, beş sayısı ve 1.0 sürümünün olasılıklarıyla birlikte (bir kez hesaplanır, yaklaşık yirmi saniye sürer)."""
    if "_course_cache" in globals():
        return _course_cache
    SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
    episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
    rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                            for record in afdb.RECORDS for start, end, name in episodes[record]])
    hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
    hours["set"] = hours.index.map(SPLIT)

    WINDOW_S = 60


    def minutes_of(beat_s, episodes=None, record=""):
        """Bir kaydın bir dakikalık pencereleri: Her dakikada biten RR aralıkları ve ritim biliniyorsa dakikanın AF içinde geçen payı."""
        rr = np.diff(beat_s)
        rr_end = beat_s[1:]
        rows = []
        for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
            x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
            if len(x) < 20 or x.max() > 3.0:
                continue
            row = {"record": record, "start_s": float(start), "rr": x}
            if episodes is not None:
                af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
                row["af_share"] = af_s / WINDOW_S
            rows.append(row)
        return rows

    minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
    minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
    minutes["set"] = minutes["record"].map(SPLIT)


    def sample_entropy(rr, m=1, r=0.03):
        """Örneklem entropisi: İlk m aralığı birbirine benzeyen, yani en fazla r saniye farklı olan iki dizinin m + 1 aralıkta da benzer olma olasılığının logaritmasının eksi işaretlisi."""
        n = len(rr)

        def pairs(length):
            t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
            alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
            return (alike.sum() - len(t)) / 2

        b, a = pairs(m), pairs(m + 1)
        return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


    def features(rr):
        """Bir dakikanın beş sayısı: Kalp hızı, değişim katsayısı, ortalama RR'ye bölünmüş RMSSD, pNN50 ve örneklem entropisi."""
        d = np.diff(rr)
        return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
                "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

    minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

    V1_0 = ["hr", "cv", "rmssd", "pnn50"]
    LOGGED = ["cv", "rmssd"]


    def design(table, names):
        """Modelin değişkenleri: CV'nin ve RMSSD'nin logaritması, ötekiler olduğu gibi."""
        X = table[names].astype(float).copy()
        for name in names:
            if name in LOGGED:
                X[name] = np.log(X[name])
        return X


    def fit_model(names, version, training):
        """Standartlaştırılmış değişkenlerle bir lojistik regresyon; JSON'un saklayabileceği düz bir sözlük olarak tutulur."""
        X = design(training, names)
        mean, sd = X.mean(), X.std()
        lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
        return {"name": "RR aralıklarından AF", "version": version, "features": names,
                "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
                "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


    def af_probability(model, table):
        """Bir tablonun her dakikası için AF olasılığı."""
        names = model["features"]
        z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
        w = np.array([model["coef"][name] for name in names])
        return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


    def choose_threshold(p, y, sensitivity=0.95):
        """AF dakikalarının istenen payını hâlâ bulan, 0,01 adımlı en yüksek eşik."""
        grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
        return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


    def alarm_episodes(alarm):
        """Art arda gelen alarm dakikalarının dizileri: (ilk, son) konumları."""
        runs, first = [], None
        for k, a in enumerate(alarm):
            if a and first is None:
                first = k
            if not a and first is not None:
                runs.append((first, k - 1))
                first = None
        if first is not None:
            runs.append((first, len(alarm) - 1))
        return runs


    def by_record(table, p, threshold):
        """Her kayıt için işaretlemenin ve modelin AF yükü, alarm atakları ve bunların yanlış olanları."""
        rows = []
        for record, part in table.assign(p=p).groupby("record", sort=True):
            alarm = (part["p"] >= threshold).to_numpy()
            runs = alarm_episodes(alarm)
            false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
            rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                         "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
        return pd.DataFrame(rows).set_index("record")

    training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
    model = fit_model(V1_0, "1.0.0", training)
    p_val = af_probability(model, validation)
    model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())


    def contributions(model, row):
        """Her değişkenin bir dakikadaki AF log-odds değerine kattığı: Katsayısı çarpı z-skoru."""
        x = design(pd.DataFrame([row]), model["features"]).iloc[0]
        z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
        return {name: model["coef"][name] * z[name] for name in model["features"]}


    def predict(request, model, digest):
        """Tahmin servisi: Bir kaydın vuru zamanlarını içeren istek girer, sonuç çıkar; ikisi de JSON'un taşıyabileceği sözlüklerdir."""
        table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
        table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
        p = af_probability(model, table)
        alarm = p >= model["threshold"]
        runs = alarm_episodes(alarm)
        return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
                "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
                "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

    ward = holter.ward()
    ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                                 for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
    ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
    ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))
    minutes["p10"] = af_probability(model, minutes)
    ward_minutes["p10"] = af_probability(model, ward_minutes)
    digest = hashlib.sha256(json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    globals()["_course_cache"] = dict(minutes=minutes, model=model, digest=digest, ward=ward, ward_minutes=ward_minutes,
                                      contributions=contributions, predict=predict, design=design, by_record=by_record)
    return _course_cache


def _minutes_table(columns, sets=("training", "validation", "test")):
    """Seçilen sütunlarla AFDB dakikalarının bir tablosu."""
    m = _course()["minutes"]
    m = m[m["set"].isin(sets)]
    cols = {"kayıt": m["record"], "küme": m["set"].map(_SETS), "başlangıç (s)": m["start_s"].round(2),
            "AF": m["af"], "AF payı": m["af_share"].round(3), "RR aralıkları (s)": [list(np.round(x, 3)) for x in m["rr"]],
            "kalp hızı": m["hr"].round(2), "CV": m["cv"].round(4), "RMSSD/RR": m["rmssd"].round(4), "pNN50": m["pnn50"].round(4),
            "olasılık": m["p10"].round(4)}
    return pd.DataFrame({k: v for k, v in cols.items() if k in columns}).reset_index(drop=True)



def _project_minutes():
    """AFDB'nin doğrulama dakikaları ve telemetri servisinin dakikaları; RR aralıkları, dört sayı ve 1.0 sürümünün olasılığıyla birlikte."""
    c = _course()
    v = c["minutes"][c["minutes"]["set"] == "validation"]
    w = c["ward_minutes"]
    rows = []
    for src, t in (("AFDB", v), ("servis", w)):
        rows.append(pd.DataFrame({"kaynak": src, "kayıt": t["record"].to_numpy(), "başlangıç (sa)": (t["start_s"] / 3600).round(4).to_numpy(),
                                  "AF": t["af"].to_numpy(), "olasılık, 1.0 sürümü": t["p10"].round(4).to_numpy()}))
    return pd.concat(rows, ignore_index=True)
print("Proje hazır.")

### Proje adımı 1: Kullanıcı testinin alarm atakları

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog bir AF prototipinin kullanıcı testini beş doğrulama kaydında ve bir telemetri servisinin dört hastasında yapıyor ve incelenecek alarm ataklarının, her atak için bir satır olmak üzere, listesine ihtiyaç duyuyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda kullanıcı testinin dakikaları bulunur: MIT-BIH Atriyal Fibrilasyon Veri Tabanı'nın 5 doğrulama kaydı ve sanal kliniğin telemetri servisindeki dört hastanın 24 saatlik kayıtları; servisin verileri sentetiktir. Her satır bir dakikadır: "kaynak" "AFDB" ya da "servis" değerini, "kayıt" kaydı ya da hastayı, "başlangıç (sa)" ise dakikanın kaydın başından itibaren saat cinsinden başlangıcını verir. Dakikanın yarısından fazlası işaretlemede ya da kaydın gerçeğinde atriyal fibrilasyon (AF) içindeyse "AF" 1 olur; "olasılık, 1.0 sürümü" ise dersin bir modelinin 1.0 sürümünün verdiği AF olasılığıdır; model 0,37 ya da daha yüksek olasılıkta alarm verir.  
> Görev: Her kayıt içinde dakikaları başlangıçlarına göre sırala. Bir alarm atağı, alarm veren art arda dakikalardan oluşan bir dizidir. Her alarm atağı için kaynağı, kaydı, saat cinsinden başlangıcı, dakika cinsinden uzunluğu, en yüksek olasılığı ve atağın herhangi bir dakikasında AF olup olmadığını içeren bir satır oluştur.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Tablonun ilk satırlarını ve her kaydın atak sayısını göster; toplam alarm atağı sayısını "cevap" adıyla sakla.  
> Test: Atakların uzunluklarının toplamı alarm veren dakikaların sayısına eşit olmalı.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_minutes()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Hekimin iş yükü

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog, 1.0 sürümüyle bir hekimin 24 saatte kaç alarm atağını inceleyeceğini, bunların kaçının yanlış olduğunu ve yanlış olanların hangi kayıtlarda toplandığını öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda kullanıcı testinin dakikaları bulunur: MIT-BIH Atriyal Fibrilasyon Veri Tabanı'nın 5 doğrulama kaydı ve sanal kliniğin telemetri servisindeki dört hastanın 24 saatlik kayıtları; servisin verileri sentetiktir. Her satır bir dakikadır: "kaynak" "AFDB" ya da "servis" değerini, "kayıt" kaydı ya da hastayı, "başlangıç (sa)" ise dakikanın kaydın başından itibaren saat cinsinden başlangıcını verir. Dakikanın yarısından fazlası işaretlemede ya da kaydın gerçeğinde atriyal fibrilasyon (AF) içindeyse "AF" 1 olur; "olasılık, 1.0 sürümü" ise dersin bir modelinin 1.0 sürümünün verdiği AF olasılığıdır; model 0,37 ya da daha yüksek olasılıkta alarm verir.  
> Görev: Her kayıt içinde dakikaları başlangıçlarına göre sırala. Bir alarm atağı, alarm veren art arda dakikalardan oluşan bir dizidir; içinde AF olan hiçbir dakika bulunmayan atak ise bir yanlış alarm atağıdır. Her kayıt için dakikaları, alarm ataklarını ve yanlış alarm ataklarını say; alarm ataklarını ve yanlış olanları kaydın 24 saati başına ver. Ardından aynısını AFDB ve servis için bir bütün olarak ver.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her kayıt için bir satırdan ve iki toplamdan oluşan bir tablo göster; bütün kayıtlarda yanlış alarm ataklarının payını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Bir kaydın yanlış alarm atakları alarm ataklarını aşamaz.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_minutes()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: Doğrulama kayıtlarında iyileştirilmiş bir sürüm

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kullanıcı testi yanlış alarmların çoğunu erken vuruların verdiğini göstermiştir. Kardiyolog, eğitim kayıtlarında kurulan ve yalnızca doğrulama kayıtlarında değerlendirilen iyileştirilmiş bir sürüm istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda MIT-BIH Atriyal Fibrilasyon Veri Tabanı'ndaki (AFDB) çoğu paroksismal atriyal fibrilasyonu (AF) olan kişilerin yaklaşık on saatlik 25 Holter kaydından elde edilen bir dakikalık pencereler bulunur. Test kayıtları tabloda yoktur. Her satır bir dakikadır: "kayıt" kaydı, "küme" kaydın kümesini ("eğitim", 15 kayıt; "doğrulama", 5 kayıt), "başlangıç (s)" ise dakikanın kaydın başından itibaren saniye cinsinden başlangıcını verir. Dakikanın yarısından fazlası uzmanların işaretlediği AF içindeyse "AF" 1, değilse 0 olur. "RR aralıkları (s)" sütununun her hücresi, dakikanın RR aralıklarını saniye cinsinden ve sırasıyla, 20 ya da daha fazla sayıdan oluşan bir liste olarak tutar. Dakikanın dört sayısı "kalp hızı", "CV", "RMSSD/RR" ve "pNN50" sütunlarındadır.  
> Görev: Her dakika için ayrıca RR aralıklarının örneklem entropisini bir aralıklı dizilerle ve 0,03 saniyelik bir toleransla hesapla: Dakikanın ilk n − 1 aralığı arasında en fazla 0,03 saniye farklı olan çiftleri say (B); bu çiftlerden kaçının, her aralığa bir sonraki aralık eklendiğinde de benzer kaldığını say (A). Örneklem entropisi, B'nin A'ya bölümünün doğal logaritmasıdır; A 0 ise 1 say, B 0 ise dakikaya n'nin doğal logaritmasını ver. Eğitim dakikalarında en fazla 3.000 yinelemeyle AF için beş girdili bir lojistik regresyon kur: Kalp hızı, CV'nin doğal logaritması, RMSSD/RR'nin doğal logaritması, pNN50 ve örneklem entropisi; her biri eğitim dakikalarının ortalaması ve standart sapmasıyla bir z-skoruna dönüştürülür. Eşiği yalnızca doğrulama dakikalarında seç: 0,05, 0,06, 0,07 ve bu biçimde 0,95'e kadar giden değerler arasından, AF olan doğrulama dakikalarının en az yüzde 95'inin olasılığının ona eşit ya da ondan büyük olduğu en yüksek değeri al. Doğrulama AUC değerini, eşikteki duyarlılığı ve özgüllüğü ve her doğrulama kaydının yanlış alarm ataklarını ver.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Sonuçları göster, katsayıları ve eşiği yazdır; eşiği iki ondalık basamakla "cevap" adıyla sakla.  
> Test: Hiçbir test dakikası kullanılmamalı ve doğrulama kayıtlarındaki duyarlılık en az 0,95 olmalı.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _minutes_table(["kayıt", "küme", "başlangıç (s)", "AF", "RR aralıkları (s)",
                       "kalp hızı", "CV", "RMSSD/RR", "pNN50"], sets=("training", "validation"))
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Test kayıtlarında iki sürüm ve değişiklik kaydı

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog test kayıtlarını iki sürüm için bir kez açıyor ve yeni sürümün değişiklik kaydını kazançları ve bedelleriyle birlikte yazıyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda MIT-BIH Atriyal Fibrilasyon Veri Tabanı'nın 5 test kaydının, yani çoğu paroksismal atriyal fibrilasyonu (AF) olan kişilerin Holter kayıtlarının bir dakikalık pencereleri bulunur. Her satır bir dakikadır: "kayıt" kaydı, "başlangıç (s)" dakikanın saniye cinsinden başlangıcını verir; dakikanın yarısından fazlası uzmanların işaretlediği AF içindeyse "AF" 1 olur; "AF payı" dakikanın AF içinde geçen payıdır. "RR aralıkları (s)" sütununun her hücresi dakikanın RR aralıklarını saniye cinsinden tutar; dakikanın dört sayısı "kalp hızı", "CV", "RMSSD/RR" ve "pNN50" sütunlarındadır.  
> Görev: 1.0 sürümü dört girdi kullanır: Kalp hızı, CV'nin doğal logaritması, RMSSD/RR'nin doğal logaritması ve pNN50; 1.1 sürümü bunlara RR aralıklarının bir aralıklı dizilerle ve 0,03 saniyelik toleransla hesaplanan örneklem entropisini ekler. İki sürümü 1. duraktaki ve 3. adımdaki biçimleriyle, eğitim dakikalarında kurulduğu gibi ve doğrulama dakikalarından gelen eşikleriyle kullan. Her kayıt içinde dakikaları başlangıçlarına göre sırala. Bir alarm atağı, alarm veren art arda dakikalardan oluşan bir dizidir; içinde AF olan hiçbir dakika bulunmayan atak ise bir yanlış alarm atağıdır. Her sürüm için test dakikalarında AUC değerini, duyarlılığı, özgüllüğü, yanlış alarm ataklarını ve kayıt başına AF yükündeki ortalama mutlak hatayı yüzde puan olarak hesapla. Ardından yeni sürüm için sürüm numarasını, değişikliği, gerekçeyi, eşiği, doğrulama ve test kayıtlarındaki etkileri ve neyin iyileşmediğini içeren bir değişiklik kaydı yaz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki sürümün tablosunu göster ve değişiklik kaydını yazdır; yeni sürümün test kayıtlarındaki duyarlılığını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Test kayıtları her sürüm için bir kez kullanılmalı ve onlara bakıldıktan sonra hiçbir eşik değiştirilmemeli.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _minutes_table(["kayıt", "başlangıç (s)", "AF", "AF payı", "RR aralıkları (s)",
                       "kalp hızı", "CV", "RMSSD/RR", "pNN50"], sets=("test",))
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Bu kısım Python yazmayı öğrenmek isteyen katılımcılar içindir. Kısım 1'i çalışmış olmanız gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır. Bazı alıştırma hücreleri `veri` adlı bir tabloyu hazırlayan bir satırla başlar. Bu tablo, etkileşimli labın prompt atölyesindeki alıştırma promptları için hazırlanmıştır; Python yolunda alıştırma, metninde adı geçen tablolarla çözülebilir.

## Python adımı 10: Defterden dosyalara

Bu hafta model defterden çıkar: Bir dosyada tutulur, dosyası bir parmak izi alır ve fonksiyonları bir web arayüzünün kullanabileceği bir modüle taşınır. Bu adımda bir sözlüğün JSON biçiminde bir metne nasıl dönüştüğü ve nasıl geri geldiği, `pathlib` modülünün bir dosyayı nasıl yazıp okuduğu, `hashlib` modülünün bir dosyanın parmak izini nasıl hesapladığı, `%%writefile` sihirli komutunun bir hücreyi nasıl dosyaya dönüştürdüğü ve sürüm numaralarının neden metin olarak değil, sayılardan oluşan demetler olarak karşılaştırıldığı anlatılmaktadır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
toy = {"name": "AF-RR", "version": "1.0.0", "features": ["hr", "cv"], "coef": {"hr": 2.5, "cv": 3.3}, "intercept": -3.4, "threshold": 0.37}
text = json.dumps(toy, indent=2, sort_keys=True)
print(text)
back = json.loads(text)
print(type(text).__name__, type(back).__name__, back == toy, back["coef"]["cv"])

**Python notu.** Bir sözlük başka sözlükleri ve listeleri tutabilir: `toy["coef"]` kendisi de bir sözlüktür ve `toy["coef"]["cv"]` değerlerinden birine ulaşır. `json.dumps` sözlüğü JSON biçiminde, yani her programın okuyabildiği bir metne dönüştürür. `indent=2` her girdiyi kendi satırına koyar; `sort_keys=True` ise anahtarları alfabe sırasıyla yazar ve böylece aynı sözlük her zaman aynı metni verir. `json.loads` metni yeniden bir sözlüğe dönüştürür; `back == toy` de hiçbir şeyin kaybolmadığını denetler. `type(text).__name__` bir değerin türünün adını verir: Bir metin için `str`, bir sözlük için `dict`.

In [ ]:
# Python alıştırması: Bir dosya ve parmak izi
path = pathlib.Path("toy_model.json")
path.write_text(text, encoding="utf-8")
print(path.name, path.stat().st_size, "bayt")
digest = hashlib.sha256(path.read_bytes()).hexdigest()
print("SHA-256", digest)
changed = dict(toy, threshold=0.38)
digest2 = hashlib.sha256(json.dumps(changed, indent=2, sort_keys=True).encode("utf-8")).hexdigest()
print("SHA-256", digest2)
print(sum(a == b for a, b in zip(digest, digest2)), "basamak 64 basamak içinde aynı yerde aynı")

**Python notu.** `pathlib.Path("toy_model.json")` defterin klasöründeki bir dosyayı adlandırır. `write_text` dosyaya bir metin yazar, `read_bytes` baytlarını geri okur, `stat().st_size` de dosyanın bayt cinsinden büyüklüğünü verir. `hashlib.sha256(...)` baytların parmak izini hesaplar; `hexdigest()` bu parmak izini 0-9 rakamları ve a-f harflerinden oluşan 64 onaltılık basamak olarak yazar. Bir metin `.encode("utf-8")` ile baytlara dönüşür. `dict(toy, threshold=0.38)` sözlüğün tek bir değeri değiştirilmiş bir kopyasını oluşturur; `zip(digest, digest2)` ise iki parmak izinin basamaklarını tek tek eşleştirir.

In [ ]:
%%writefile toy_tools.py
# Python alıştırması: Kendinize ait küçük bir modül
"""Deneme modelinin araçları."""


def version_tuple(version):
    """'1.10.0', (1, 10, 0) olur; böylece sürümler sayı olarak karşılaştırılabilir."""
    return tuple(int(part) for part in version.split("."))


def alarm(probability, threshold=0.37):
    """Olasılık eşiğe ulaştığında True döndürür."""
    return probability >= threshold

In [ ]:
# Python alıştırması: Modülü kullanmak
import importlib
import toy_tools
importlib.reload(toy_tools)
print(toy_tools.version_tuple("1.10.0"), toy_tools.alarm(0.40), toy_tools.alarm(0.40, threshold=0.5))
print("metin olarak:", "1.10.0" > "1.9.0", "| demet olarak:", toy_tools.version_tuple("1.10.0") > toy_tools.version_tuple("1.9.0"))

**Python notu.** `%%writefile toy_tools.py` ile başlayan bir hücre çalıştırılmaz: Jupyter hücrenin metnini `toy_tools.py` dosyasına yazar ve bu dosya artık küçük bir *modül* tutar. `import toy_tools` modülü okur; `toy_tools.alarm(...)` da fonksiyonlarından birini çağırır. `importlib.reload`, dosya değiştikten sonra onu yeniden okur; bu olmadan Python ilk okuduğu sürümü tutar. `alarm` fonksiyonunun `threshold` için bir *varsayılan değeri* vardır: Çağrıda verilmezse değeri 0,37 olur. Bir `def` satırının altında üç tırnak içinde yazılan metin fonksiyonun *belge metnidir* (docstring); `help(toy_tools.alarm)` bu metni gösterir. Metinler harf harf karşılaştırıldığı için "1.10.0" metni "1.9.0" metninden önce gelir; sayılardan oluşan demetler ise sayı sayı karşılaştırılır ve (1, 10, 0), (1, 9, 0) demetinden sonra gelir.

**Hızlı kontrol.** Bu haftanın model dosyası yazılırken neden `sort_keys=True` kullanılır?

<details><summary>Cevap</summary>

Aynı modelin her zaman aynı metni, dolayısıyla aynı parmak izini vermesi için. Bu olmadan anahtarları başka bir sırayla yazan iki program tek bir model için iki farklı dosya ve iki farklı parmak izi üretirdi.

</details>

**Alıştırma 10.1.** `toy` sözlüğünü `indent` olmadan ve anahtarları sıralı olarak JSON biçiminde yazın. Metin kaç karakterden oluşur?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 10.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = len(json.dumps(toy, sort_keys=True))
teach.check("Alıştırma 10.1", cevap)

**Alıştırma 10.2.** Sabit terimi −3,41 olan bir `toy` kopyası oluşturun, ikisini de `indent=2` ve sıralı anahtarlarla JSON biçiminde yazın ve SHA-256 parmak izlerini hesaplayın. İki parmak izi 64 yerden kaçında aynı basamağı taşır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 10.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
a = hashlib.sha256(json.dumps(toy, indent=2, sort_keys=True).encode("utf-8")).hexdigest()
b = hashlib.sha256(json.dumps(dict(toy, intercept=-3.41), indent=2, sort_keys=True).encode("utf-8")).hexdigest()
cevap = sum(x == y for x, y in zip(a, b))
teach.check("Alıştırma 10.2", cevap)

**Alıştırma 10.3.** "1.2.0", "1.10.0", "1.9.3" ve "1.10.1" sürümlerini `toy_tools.version_tuple` ile en eskiden en yeniye sıralayın. 1'den başlayarak sayıldığında "1.9.3" kaçıncı sıradadır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 10.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
order = sorted(["1.2.0", "1.10.0", "1.9.3", "1.10.1"], key=toy_tools.version_tuple)
cevap = order.index("1.9.3") + 1
teach.check("Alıştırma 10.3", cevap)

## 1. Kayıtlar ve ritimleri

*Ders: [RR aralıklarından AF: Bir dakikanın beş sayısı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#af-from-rr-intervals-five-numbers-of-a-minute)*

MIT-BIH Atriyal Fibrilasyon Veri Tabanı yaklaşık on saatlik 25 Holter kaydı içerir. Uzmanlar ritmi dört etiketle elle işaretlemiştir: Atriyal fibrilasyon için `(AFIB`, atriyal flatter için `(AFL`, AV kavşak ritmi için `(J` ve öteki bütün ritimler için `(N`. Baştaki parantez, işaretleme dosyalarında her etiketin bir parçasıdır. Vurular ise bir servisteki kayıt cihazında olduğu gibi otomatik bir QRS dedektörüyle bulunmuştur. Ders paketi işaretlemeleri `afdb.rhythm` ile okur; bu fonksiyon her kaydın ataklarını saniye cinsinden (başlangıç, bitiş, ritim) olarak verir. `afdb.SPLITS` ise dersin kayıt düzeyindeki ayırmasını tutar.

**Dersten koda.** Kayıt düzeyinde ayırma: `SPLIT`. Her kaydın ritim atakları: `episodes`. Kayıt başına her ritmin saati ve her kaydın kümesi: `hours`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                        for record in afdb.RECORDS for start, end, name in episodes[record]])
hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
hours["set"] = hours.index.map(SPLIT)

print(hours.groupby("set")[["(AFIB", "(AFL", "(J", "(N"]].sum().round(1))

**Python notu.** `afdb.SPLITS` bir kümenin adından kayıtlarına giden bir sözlüktür; ilk satırdaki *sözlük üreteci* (dictionary comprehension) bu sözlüğü tersine çevirir: `SPLIT["08219"]`, 08219 kaydının kümesini verir. `episodes` her kaydın atak listesini tutar. `pd.DataFrame([...])` içindeki liste üreteci her atak için bir satır oluşturur ve atağın saat cinsinden uzunluğunu verir. `pivot_table` ardından her kayıt için her ritmin saatlerini toplar: Her kayıt için bir satır, her ritim için bir sütun; bir kayıtta bir ritmin hiç atağı yoksa `fill_value=0.0` kullanılır. `groupby("set")` her kümenin satırlarını toplar.

In [ ]:
order = hours.sort_values(["set", "(AFIB"]).index
fig, ax = plt.subplots(figsize=(9.6, 3.2))
bottom = np.zeros(len(order))
for rhythm, colour in (("(AFIB", "#9B1B1B"), ("(AFL", "#B97813"), ("(J", "#0E7A78"), ("(N", "#C8C0B8")):
    ax.bar(range(len(order)), hours.loc[order, rhythm], bottom=bottom, color=colour, label=rhythm)
    bottom += hours.loc[order, rhythm].to_numpy()
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{r}\n{hours.loc[r, 'set'][:3]}" for r in order], fontsize=7)
ax.set_ylabel("saat")
ax.legend(fontsize=8, ncol=4)
ax.set_title("AFDB'nin 25 kaydında her ritmin saati (MIT-BIH AFDB, PhysioNet, ODC-By 1.0)", fontsize=10)
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** AF'nin payı kayıttan kayda çok değişir: Bazı kayıtlarda yalnızca birkaç dakika, bazılarında ise neredeyse kaydın tamamı AF'dir. Ders verileri neden dakika düzeyinde değil de kayıt düzeyinde ayırır ve dakika düzeyinde bir ayırma test AUC değerine ne yapardı?

**Alıştırma A.** Doğrulama kayıtlarında `(AFIB` olarak işaretlenmiş kaç saat AF vardır? Sayıyı bir ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("A")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(hours.loc[hours["set"] == "validation", "(AFIB"].sum()), 1)
teach.check("Alıştırma A", cevap)

## 2. Bir dakikalık pencerelerde RR aralıkları

*Ders: [RR aralıklarından AF: Bir dakikanın beş sayısı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#af-from-rr-intervals-five-numbers-of-a-minute)*

Prototip her dakikayı değerlendirir. `minutes_of` bir kaydın vurularını bir dakikalık pencerelere böler ve her pencerede biten RR aralıklarını tutar. 20'den az aralığı olan ya da 3 saniyeden uzun bir aralık içeren dakikalar dışarıda bırakılır. Ritim biliniyorsa fonksiyon dakikanın AF içinde geçen payını da verir; bu pay yarıdan büyükse dakika AF sayılır.

**Dersten koda.** Bir dakikanın $RR_i$ aralıkları: `x`. Dakikanın AF içinde geçen payı: `af_share`. Dakikanın etiketi, AF için 1: `af`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
WINDOW_S = 60


def minutes_of(beat_s, episodes=None, record=""):
    """Bir kaydın bir dakikalık pencereleri: Her dakikada biten RR aralıkları ve ritim biliniyorsa dakikanın AF içinde geçen payı."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows

minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
minutes["set"] = minutes["record"].map(SPLIT)

print(len(minutes), "dakika")
print(minutes.groupby("set")["af"].agg(["count", "sum"]).rename(columns={"count": "dakika", "sum": "AF dakikası"}))

**Python notu.** `np.diff(beat_s)` ardışık vurular arasındaki aralıkları, `beat_s[1:]` ise her aralığın bittiği zamanı verir. `np.arange(first, last, WINDOW_S)` her pencerenin başlangıcını verir. Döngünün içindeki `(rr_end >= start) & (rr_end < start + WINDOW_S)` *maskesi* pencerede biten aralıkları tutar. `continue`, dışarıda bırakılan bir dakika için döngünün geri kalanını atlar. AF payı, `(AFIB` olarak işaretlenmiş her atağın pencereyle örtüşen kısmını toplar: `max(0.0, min(e, start + 60) - max(s, start))`. `astype(int)` True ve False değerlerini 1 ve 0'a dönüştürür.

In [ ]:
record = "04126"
part = minutes[minutes["record"] == record]
pair = [part[part["af"] == 0].iloc[0], part[part["af"] == 1].iloc[0]]
fig, axes = plt.subplots(1, 2, figsize=(9.6, 2.8), sharey=True)
for ax, row, name in zip(axes, pair, ["AF yok", "AF"]):
    ax.plot(row["rr"], marker="o", ms=3, color="#9B1B1B" if row["af"] else "#1F5F8B")
    ax.set_title(f"{record}, {row['start_s'] / 3600:.2f} h: {name}", fontsize=10)
    ax.set_xlabel("dakikanın aralığı")
axes[0].set_ylabel("RR (s)")
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** AF olmayan dakikada aralıklar yavaşça değişir; AF olan dakikada ise her aralık bir öncekinden farklıdır. Art arda gelen atriyal erken vurular aynı grafikte nasıl görünürdü?

**Alıştırma B.** Test kayıtlarının kaç dakikası AF sayılır?

In [ ]:
veri = alistirma_tablosu("B")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(minutes.loc[minutes["set"] == "test", "af"].sum())
teach.check("Alıştırma B", cevap)

## 3. Bir dakikanın beş sayısı

*Ders: [RR aralıklarından AF: Bir dakikanın beş sayısı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#af-from-rr-intervals-five-numbers-of-a-minute)*

Dört sayı bir dakikanın aralıklarının ne kadar değiştiğini ölçer: Kalp hızı, değişim katsayısı, ortalama aralığa bölünmüş RMSSD ve pNN50. Beşincisi olan örneklem entropisi ise değişimin bir örüntüyü tekrarlayıp tekrarlamadığını ölçer.

**Formüllerden koda.** $\mathrm{HR} = 60/\overline{RR}$: `60 / rr.mean()`. $\mathrm{CV} = s_{RR}/\overline{RR}$: `rr.std(ddof=1) / rr.mean()`. $\mathrm{RMSSD}/\overline{RR}$: Ardışık farklar `d` olmak üzere `np.sqrt(np.mean(d ** 2)) / rr.mean()`. pNN50: `np.mean(np.abs(d) > 0.05)`. `sample_entropy` içinde $B$ ve $A$ sayıları: `b` ve `a`; $m$ = `m` ve $r$ = `r`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
def sample_entropy(rr, m=1, r=0.03):
    """Örneklem entropisi: İlk m aralığı birbirine benzeyen, yani en fazla r saniye farklı olan iki dizinin m + 1 aralıkta da benzer olma olasılığının logaritmasının eksi işaretlisi."""
    n = len(rr)

    def pairs(length):
        t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
        alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
        return (alike.sum() - len(t)) / 2

    b, a = pairs(m), pairs(m + 1)
    return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


def features(rr):
    """Bir dakikanın beş sayısı: Kalp hızı, değişim katsayısı, ortalama RR'ye bölünmüş RMSSD, pNN50 ve örneklem entropisi."""
    d = np.diff(rr)
    return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
            "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

for name, rr in (("sinüs ritmi", [0.80, 0.81, 0.82, 0.81, 0.80, 0.81, 0.82, 0.81, 0.80]),
                 ("atriyal erken vurular", [0.80, 0.81, 0.55, 1.05, 0.80, 0.81, 0.56, 1.04, 0.80]),
                 ("AF", [0.61, 0.53, 0.84, 0.62, 0.85, 0.60, 0.97, 0.83, 0.99])):
    f = features(np.array(rr))
    print(f"{name:28}", {k: round(v, 3) for k, v in f.items()})

**Python notu.** `np.lib.stride_tricks.sliding_window_view(rr, length)`, art arda gelen `length` aralıktan oluşan her diziyi bir tablonun bir satırı olarak verir; `[: n - m]` de iki uzunluk için aynı sayıda, yani ilk $n - m$ diziyi tutar. `t[:, None, :] - t[None, :, :]` her diziyi bir kerede öteki her diziyle karşılaştırır: dizi × dizi × uzunluk büyüklüğünde bir farklar tablosu. `.max(axis=2) <= r`, her çift için bütün aralıklarının $r$ içinde kalıp kalmadığını söyler. Bir dizinin kendisiyle oluşturduğu çiftler köşegendedir; `alike.sum() - len(t)` bunları çıkarır, `/ 2` de her çifti bir kez sayar. Üç kısa dakika, dersteki çözümlü örneğin dakikalarıdır.

In [ ]:
minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

train = minutes[minutes["set"] == "training"]
print(train.groupby("af")[["hr", "cv", "rmssd", "pnn50", "sampen"]].median().round(3))

In [ ]:
names = {"hr": "kalp hızı (dakikada)", "cv": "CV", "rmssd": "RMSSD/RR", "pnn50": "pNN50", "sampen": "örneklem entropisi"}
fig, axes = plt.subplots(1, 5, figsize=(11, 2.8))
for ax, (key, label) in zip(axes, names.items()):
    ax.boxplot([train.loc[train["af"] == 0, key], train.loc[train["af"] == 1, key]], showfliers=False)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["AF yok", "AF"])
    ax.set_title(label, fontsize=9)
    if key in ("cv", "rmssd"):
        ax.set_yscale("log")
plt.suptitle("Eğitim dakikalarının beş sayısı (MIT-BIH AFDB, PhysioNet, ODC-By 1.0)", fontsize=10)
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** CV, RMSSD/RR ve pNN50 kutuları neredeyse hiç örtüşmez; kalp hızının kutuları ise çok örtüşür. Düzensizliği ölçen üç sayının üst bıyıklarında AF olmayan hangi dakikaları görmeyi beklerdiniz?

**İleri düzey (isteğe bağlı).** 0,03 saniyelik $r$ toleransı saniye cinsinden sabittir. Lake ve Moorman çok kısa seriler için ortalama aralığa göre belirlenen toleransları da incelemiştir. Üç kısa dakikada `sample_entropy(x, r=0.02)` ve `r=0.05` değerlerini deneyin ve hangi sayıların değiştiğini görün.

**Alıştırma C.** AF olan eğitim dakikalarının ortanca örneklem entropisi nedir? İki ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("C")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(train.loc[train["af"] == 1, "sampen"].median()), 2)
teach.check("Alıştırma C", cevap)

## 4. Model, eşiği ve AF yükü

*Ders: [Olasılıklardan alarmlara ve AF yüküne](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#from-probabilities-to-alarms-and-the-af-burden)*

Modelin 1.0 sürümü dört sayıyla kurulan bir lojistik regresyondur: Kalp hızı, CV ve RMSSD/RR değerlerinin logaritmaları ve pNN50; her biri eğitim dakikalarına göre bir z-skoru olarak girer. Eşiği, doğrulama kayıtlarındaki AF dakikalarının en az yüzde 95'inin alarm verdiği en yüksek eşiktir. Art arda gelen alarm dakikaları alarm ataklarını, alarm veren dakikaların payı ise AF yükünü verir.

**Formüllerden koda.** $g(x_j)$: `design`. $\mu_j$ ve $\sigma_j$: `fit_model` içinde `mean` ve `sd`. $z_j$ ve $p = 1/(1 + e^{-\eta})$: `af_probability`. $t$ eşiği: `choose_threshold`. Alarm atakları: `alarm_episodes`. Her kaydın AF yükü ve yanlış alarm atakları: `by_record`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
V1_0 = ["hr", "cv", "rmssd", "pnn50"]
LOGGED = ["cv", "rmssd"]


def design(table, names):
    """Modelin değişkenleri: CV'nin ve RMSSD'nin logaritması, ötekiler olduğu gibi."""
    X = table[names].astype(float).copy()
    for name in names:
        if name in LOGGED:
            X[name] = np.log(X[name])
    return X


def fit_model(names, version, training):
    """Standartlaştırılmış değişkenlerle bir lojistik regresyon; JSON'un saklayabileceği düz bir sözlük olarak tutulur."""
    X = design(training, names)
    mean, sd = X.mean(), X.std()
    lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
    return {"name": "RR aralıklarından AF", "version": version, "features": names,
            "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
            "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


def af_probability(model, table):
    """Bir tablonun her dakikası için AF olasılığı."""
    names = model["features"]
    z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
    w = np.array([model["coef"][name] for name in names])
    return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


def choose_threshold(p, y, sensitivity=0.95):
    """AF dakikalarının istenen payını hâlâ bulan, 0,01 adımlı en yüksek eşik."""
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


def alarm_episodes(alarm):
    """Art arda gelen alarm dakikalarının dizileri: (ilk, son) konumları."""
    runs, first = [], None
    for k, a in enumerate(alarm):
        if a and first is None:
            first = k
        if not a and first is not None:
            runs.append((first, k - 1))
            first = None
    if first is not None:
        runs.append((first, len(alarm) - 1))
    return runs


def by_record(table, p, threshold):
    """Her kayıt için işaretlemenin ve modelin AF yükü, alarm atakları ve bunların yanlış olanları."""
    rows = []
    for record, part in table.assign(p=p).groupby("record", sort=True):
        alarm = (part["p"] >= threshold).to_numpy()
        runs = alarm_episodes(alarm)
        false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
        rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                     "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
    return pd.DataFrame(rows).set_index("record")

training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
model = fit_model(V1_0, "1.0.0", training)
p_val = af_probability(model, validation)
model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())

print("katsayılar", model["coef"], "| sabit terim", model["intercept"])
print("doğrulama AUC", round(roc_auc_score(validation["af"], p_val), 3), "| eşik", model["threshold"])
records_val = by_record(validation, p_val, model["threshold"])
print(records_val.round(1))
print("alarm atakları", int(records_val["episodes"].sum()), "| yanlış", int(records_val["false_episodes"].sum()))

**Python notu.** Model düz bir sözlüktür ve adını, sürümünü, değişkenlerini, ortalamalarını, standart sapmalarını, katsayılarını ve sabit terimini tutar. Sayılar altı ondalık basamağa yuvarlanmıştır; böylece 6. bölümün dosyası kısa ve okunur kalır. `(X - mean) / sd`, pandas sütunları adlarıyla eşleştirdiği için her sütunu bir kerede z-skorlarına dönüştürür. `af_probability` içinde `z[names].to_numpy() @ w`, z-skorlarının her satırını katsayılarla çarpar ve toplar: `@` bir matris ile bir vektörün çarpımıdır. `choose_threshold` içinde `max(t for t in grid if ...)` ızgaradaki eşiklerden duyarlılığı koruyan en büyüğünü alır. `groupby("record", sort=True)` kayıtları tek tek dolaşır; `alarm_episodes` da her kaydın dakikalarını gezerek dizileri bulur.

In [ ]:
part = validation[validation["record"] == "08219"]
p = af_probability(model, part)
fig, ax = plt.subplots(figsize=(9.6, 2.8))
ax.fill_between(part["start_s"] / 3600, 0, part["af"], color="#C8C0B8", step="post", label="işaretlemede AF")
ax.plot(part["start_s"] / 3600, p, color="#9B1B1B", lw=0.9, label="olasılık, 1.0 sürümü")
ax.axhline(model["threshold"], color="#1F5F8B", ls="--", lw=0.9, label="eşik")
ax.set_xlabel("08219 kaydının başından itibaren saat")
ax.set_ylabel("AF olasılığı")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** 08219 kaydı gri AF atakları dışında çok sayıda alarm verir. Herhangi bir EKG şeridine bakmadan önce bu dakikalar hakkında ne sorardınız?

**Alıştırma D.** `by_record` fonksiyonunu doğrulama kayıtlarına modelin eşiği yerine 0,45 eşiğiyle uygulayın. Kaç yanlış alarm atağı kalır?

In [ ]:
veri = alistirma_tablosu("D")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(by_record(validation, p_val, 0.45)["false_episodes"].sum())
teach.check("Alıştırma D", cevap)

## 5. Neden alarm verdi? Açıklama ve EKG şeridi

*Ders: [Neden alarm verdi? Açıklama ve EKG şeridi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#why-did-it-alarm-the-explanation-and-the-strip)*

Bir lojistik regresyonda bir dakikanın log-odds değeri, sabit terime her sayı için bir terimin eklenmesiyle bulunur ve her terim o sayının kattığı değerdir. Bu bölüm 08219 kaydının zaman sırasıyla dizilmiş yanlış alarm dakikalarından ortadakini açıklar ve bu dakikanın on saniyelik EKG'sini RR aralıklarının yanında gösterir. Bu dakika, dersteki dakikadır.

**Formüllerden koda.** $c_j = \beta_j z_j$ katkısı: `contributions(model, row)`. $\eta = \beta_0 + \sum_j c_j$ log-odds değeri: `model["intercept"] + sum(c.values())`. EKG şeridi: `afdb.strip(record, start_s)`; iki EKG sinyali milivolt cinsinden.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
def contributions(model, row):
    """Her değişkenin bir dakikadaki AF log-odds değerine kattığı: Katsayısı çarpı z-skoru."""
    x = design(pd.DataFrame([row]), model["features"]).iloc[0]
    z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
    return {name: model["coef"][name] * z[name] for name in model["features"]}

false_08219 = validation[(validation["record"] == "08219") & (validation["af"] == 0)]
false_alarms = false_08219[af_probability(model, false_08219) >= model["threshold"]]
row = false_alarms.iloc[len(false_alarms) // 2]
c = contributions(model, row)
eta = model["intercept"] + sum(c.values())
print(f"dakika: {row['start_s'] / 3600:.2f} h", {k: round(v, 3) for k, v in row[["hr", "cv", "rmssd", "pnn50"]].items()})
print({k: round(v, 2) for k, v in c.items()}, "| log-odds", round(eta, 2), "| olasılık", round(1 / (1 + np.exp(-eta)), 3))

**Python notu.** `false_08219`, 08219 kaydının AF olmayan dakikalarını; `false_alarms` ise bunlar arasından olasılığı eşiğe ulaşanları zaman sırasıyla tutar. `len(false_alarms) // 2` ortadaki konumdur; `.iloc` da bir konumdaki satırı alır. `contributions`, `pd.DataFrame([row])` ile tek satırlık bir tablo kurar; böylece logaritmaları bütün tablodaki ile aynı `design` fonksiyonu alır. Son satırdaki *sözlük üreteci* (dictionary comprehension), yazdırmak için `c` sözlüğünün her değerini yuvarlar.

In [ ]:
t, ecg = afdb.strip("08219", row["start_s"] + 20.0)
beats = afdb.beats("08219")
fig, axes = plt.subplots(3, 1, figsize=(9.6, 6.2), gridspec_kw={"height_ratios": [2, 1.2, 1.2]})
axes[0].plot(t - t[0], ecg[:, 0], color="#231A1A", lw=0.7)
for b in beats[(beats >= t[0]) & (beats < t[-1])]:
    axes[0].axvline(b - t[0], color="#B97813", lw=0.5, alpha=0.6)
axes[0].set_title("08219 kaydı, yanlış alarm dakikasından on saniye (MIT-BIH AFDB, PhysioNet, ODC-By 1.0)", fontsize=10)
axes[0].set_xlabel("s")
axes[0].set_ylabel("mV")
axes[1].plot(row["rr"], marker="o", ms=3, color="#9B1B1B")
axes[1].set_ylabel("RR (s)")
axes[1].set_xlabel("dakikanın aralığı")
axes[2].barh(list(c), list(c.values()), color=["#9B1B1B" if v > 0 else "#1F5F8B" for v in c.values()])
axes[2].axvline(0, color="grey", lw=0.8)
axes[2].set_xlabel("AF'nin log-odds değerine katkı")
plt.tight_layout()
plt.show()

**Python notu.** `afdb.strip`, bir kaydın sinyal dosyasından on saniyelik örnekleri okur ve her örneğin zamanını ve iki EKG sinyalini verir; bu yüzden `ecg[:, 0]` ilk sinyaldir. Turuncu çizgiler veri tabanının otomatik QRS dedektörünün bulduğu vuruları işaretler. `gridspec_kw={"height_ratios": [2, 1.2, 1.2]}` üç panele farklı yükseklikler verir. Çubuk grafikte bir liste üreteci pozitif katkı için kırmızıyı, negatif katkı için maviyi seçer.

**Gözlemle ve düşün.** EKG şeridine bakın: Hangi vurular erken geliyor, P dalgaları var mı ve aralarındaki vurular sinüs vuruları mı? Bu alarmı reddetmenin gerekçesi olarak karar kaydına ne yazardınız?

**Alıştırma E.** 08219 kaydının AF dakikaları arasından olasılığı en yüksek alarm dakikasını alın ve katkılarını hesaplayın. pNN50'nin katkısı nedir? İki ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("E")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
af_08219 = validation[(validation["record"] == "08219") & (validation["af"] == 1)]
best = af_08219.index[af_probability(model, af_08219).argmax()]
cevap = round(float(contributions(model, af_08219.loc[best])["pnn50"]), 2)
teach.check("Alıştırma E", cevap)

## 6. Model dosyası, parmak izi ve model kartı

*Ders: [Sürümler, parmak izleri ve model kartı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#versions-fingerprints-and-the-model-card)*

Model defterden JSON biçiminde bir dosya olarak çıkar ve dosyanın SHA-256 değeri modeli adlandırır. `save_model` dosyayı yazar ve parmak izini döndürür; `load_model` dosyayı geri okur ve parmak izini yeniden hesaplar; böylece prototip parmak izini gösterebilir. Model kartı modeli dokuz bölümde tarif eder.

**Dersten koda.** 1.0 sürümünün dosyası: `af_model_v1.0.0.json`. Parmak izi: `digest`. Model kartı: `card`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
def save_model(model, path):
    """Modeli JSON olarak yazar ve dosyanın parmak izi olan SHA-256 değerini döndürür."""
    text = json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False)
    pathlib.Path(path).write_text(text, encoding="utf-8")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def load_model(path):
    """save_model ile yazılmış bir modeli okur; modeli ve dosyanın SHA-256 değerini döndürür."""
    text = pathlib.Path(path).read_text(encoding="utf-8")
    return json.loads(text), hashlib.sha256(text.encode("utf-8")).hexdigest()

digest = save_model(model, "af_model_v1.0.0.json")
loaded, digest_again = load_model("af_model_v1.0.0.json")
print(pathlib.Path("af_model_v1.0.0.json").stat().st_size, "bayt · SHA-256", digest)
print("aynı model:", loaded == model, "| aynı parmak izi:", digest == digest_again)

**Python notu.** `save_model` sözlüğü `indent=2`, sıralı anahtarlar ve ı ile ş gibi harfleri olduğu gibi bırakan `ensure_ascii=False` ile yazar ve tam olarak yazdığı metnin parmak izini hesaplar. `load_model` iki değer döndürür: Model ve parmak izi; `loaded, digest_again = ...` satırı bu iki değeri ayırır. `loaded == model` iki sözlüğü, içlerindeki sözlükler de dahil olmak üzere girdi girdi karşılaştırır.

In [ ]:
v, t = validation, model["threshold"]
card = {
    "Model ayrıntıları": f"{model['name']}, sürüm {model['version']}, lojistik regresyon: {', '.join(model['features'])}; SHA-256 {digest[:12]}…",
    "Kullanım amacı": "Telemetrideki erişkinlerin RR aralıklarında olası AF dakikalarını, hekimin EKG ile incelemesi için işaretler.",
    "Etkenler": "Erken vurular, elektrot gürültüsü, atriyal flatter, QRS dedektörünün kaçırdığı ya da eklediği vurular.",
    "Ölçütler": "Dakika başına AUC, duyarlılık, özgüllük ve pozitif öngörü değeri; kayıt başına alarm atakları ve AF yükündeki hata.",
    "Eğitim verileri": f"{training['record'].nunique()} AFDB kaydı, {len(training)} dakika",
    "Değerlendirme verileri": f"doğrulama {v['record'].nunique()} kayıt ({len(v)} dakika), test {test['record'].nunique()} kayıt, bir kez açılır",
    "Nicel analizler": f"doğrulama AUC {roc_auc_score(v['af'], p_val):.3f}, duyarlılık {(p_val[v['af'] == 1] >= t).mean():.3f}, özgüllük {(p_val[v['af'] == 0] < t).mean():.3f}",
    "Etik değerlendirmeler": "Tek bir hastaneden 25 kişi; yaşa ve cinsiyete göre ve Türkiye'deki hastalarda başarım bilinmiyor.",
    "Uyarılar ve öneriler": "Yanlış alarmların çoğu erken vurulardan kaynaklanır; her alarm EKG şeridine bakılarak değerlendirilir.",
}
for section, text in card.items():
    print(f"{section}: {text}")

**Python notu.** Model kartı, Mitchell ve arkadaşlarının dokuz bölümünden metinlerine giden bir sözlüktür. *f-string* ifadeleri hesaplanan değerleri metnin içine koyar: `{roc_auc_score(...):.3f}` AUC değerini üç ondalık basamakla, `{digest[:12]}` ise parmak izinin ilk on iki basamağını yazar. `nunique()` bir tablodaki farklı kayıtları sayar. Nicel analizlerde `p_val[v['af'] == 1] >= t` AF dakikalarını tutar ve bunlardan hangilerinin eşiğe ulaştığını sorar; True ve False değerlerinin ortalaması duyarlılıktır.

**Gözlemle ve düşün.** Kart modelin ne için olduğunu ve ne için olmadığını söyler. Prototipin kendi servisinizde çalışmasına izin vermeden önce kartın hangi satırını ilk okurdunuz ve hangi satır sizin hastalarınız için eksiktir?

**Alıştırma F.** `model` sözlüğünün eşiği modelin eşiği yerine 0,38 olan bir kopyasını oluşturun, kopyayı `af_model_test_copy.json` olarak kaydedin ve doğrulama dakikalarının olasılıklarını kopyayla hesaplayın. Kaç doğrulama dakikası alarmını kaybeder?

In [ ]:
veri = alistirma_tablosu("F")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
copy_ = dict(model, threshold=0.38)
save_model(copy_, "af_model_test_copy.json")
p_copy = af_probability(copy_, validation)
cevap = int(((p_val >= model["threshold"]) & (p_copy < copy_["threshold"])).sum())
teach.check("Alıştırma F", cevap)

## 7. HL7 FHIR kaynakları olarak rapor

*Ders: [HL7 FHIR: Başka sistemlerin okuyabileceği bir rapor](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#hl7-fhir-a-report-that-other-systems-can-read)*

Prototipin raporu dört FHIR kaynağından oluşan bir Bundle'dır: Model için bir Device, AF yükü için bir Observation, bir DiagnosticReport ve bir Provenance. Rapor, bir hekim bulguyu onaylayana ya da reddedene kadar ön rapor olarak kalır. Bu bölümün hastası, 24 saati ders paketinden gelen telemetri servisindeki VC-0128'dir.

**Dersten koda.** Ön ya da kesin durum: `status`. UCUM'un "%" birimiyle yük: `valueQuantity`. SNOMED CT'de AF kodu, 49436004: `conclusionCode`. Yazar ve doğrulayıcı: Provenance kaynağının etmenleri (agent). Denetimler: `check_bundle`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
BASE_URL = "http://example.org/fhir"


def fhir_bundle(patient_id, result, model, digest, issued, decision=None, physician=None):
    """Bir kaydın raporu, HL7 FHIR R4 kaynakları olarak: Model bir Device, AF yükü bir Observation, ayrıca bir DiagnosticReport ve bir Provenance. Rapor, bir hekim onaylayana ya da reddedene kadar ön rapordur."""
    status = "final" if decision in ("confirmed", "rejected") else "preliminary"
    device = {"resourceType": "Device", "id": "af-rr",
              "deviceName": [{"name": model["name"], "type": "model-name"}],
              "version": [{"value": model["version"]}],
              "note": [{"text": "SHA-256 " + digest}]}
    burden = {"resourceType": "Observation", "id": "af-burden", "status": status,
              "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "procedure"}]}],
              "code": {"coding": [{"system": "http://example.org/vclinic", "code": "af-burden"}], "text": "AF yükü"},
              "subject": {"reference": "Patient/" + patient_id},
              "valueQuantity": {"value": result["burden_percent"], "unit": "%", "system": "http://unitsofmeasure.org", "code": "%"},
              "device": {"reference": "Device/af-rr"}}
    if result.get("start"):
        burden["effectivePeriod"] = {"start": result["start"], "end": result["end"]}
    report = {"resourceType": "DiagnosticReport", "id": "af-report", "status": status,
              "code": {"text": "Holter kaydı, AF analizi"},
              "subject": {"reference": "Patient/" + patient_id}, "issued": issued,
              "result": [{"reference": "Observation/af-burden"}],
              "conclusion": result["text"]}
    agents = [{"type": {"text": "author"}, "who": {"reference": "Device/af-rr"}}]
    if status == "final":
        report["performer"] = [{"reference": "Practitioner/" + physician}]
        agents.append({"type": {"text": "verifier"}, "who": {"reference": "Practitioner/" + physician}})
        if decision == "confirmed" and result["burden_percent"] > 0:
            report["conclusionCode"] = [{"coding": [{"system": "http://snomed.info/sct", "code": "49436004",
                                                      "display": "Atrial fibrillation (disorder)"}]}]
    provenance = {"resourceType": "Provenance", "id": "af-provenance", "target": [{"reference": "DiagnosticReport/af-report"}],
                  "recorded": issued, "agent": agents}
    resources = [device, burden, report, provenance]
    return {"resourceType": "Bundle", "type": "collection",
            "entry": [{"fullUrl": f"{BASE_URL}/{r['resourceType']}/{r['id']}", "resource": r} for r in resources]}


def check_bundle(bundle):
    """Bir paketin basit denetimleri: Her kaynağın bir türü ve bir kimliği vardır, paketin içindeki her başvuru bir kaynağa ulaşır ve yükün bir UCUM birimi vardır. Patient ve Practitioner hastanenin sisteminde bulunur."""
    problems = []
    present = {f"{e['resource']['resourceType']}/{e['resource'].get('id')}" for e in bundle["entry"]}
    for e in bundle["entry"]:
        r = e["resource"]
        if not r.get("id"):
            problems.append(r["resourceType"] + ": kimlik yok")
        for ref in re.findall(r'"reference": "([^"]+)"', json.dumps(r)):
            if ref.split("/")[0] not in ("Patient", "Practitioner") and ref not in present:
                problems.append("başvuru bir kaynağa ulaşmıyor: " + ref)
        if r["resourceType"] == "Observation":
            q = r.get("valueQuantity", {})
            if q.get("system") != "http://unitsofmeasure.org":
                problems.append("Observation: birim UCUM değil")
    return problems

rec_0128 = holter.recording()
t0128 = pd.DataFrame(minutes_of(rec_0128["detected_s"]))
t0128 = t0128.join(pd.DataFrame([features(x) for x in t0128["rr"]], index=t0128.index))
alarm_0128 = af_probability(model, t0128) >= model["threshold"]
runs_0128 = alarm_episodes(alarm_0128)
result = {"start": "2026-11-18T08:00:00+03:00", "end": "2026-11-19T08:00:00+03:00", "burden_percent": round(100 * float(alarm_0128.mean()), 1),
          "text": f"AF yükü {100 * alarm_0128.mean():.1f} %, {len(runs_0128)} alarm atağı; model {model['version']}."}
bundle = fhir_bundle("VC-0128", result, model, digest, "2026-11-19T09:30:00+03:00")
print("sorunlar:", check_bundle(bundle))
print(json.dumps(bundle["entry"][1]["resource"], indent=2, ensure_ascii=False))

**Python notu.** `holter.recording()` VC-0128'in 24 saatini verir: Gerçek vurular, kayıt cihazının QRS dedektörünün bulduğu vurular ve AF atakları. Prototip, serviste olduğu gibi yalnızca `detected_s` değerlerini görür. `fhir_bundle` kaynakları iç içe sözlükler olarak kurar; `json.dumps(..., indent=2, ensure_ascii=False)` de bunlardan birini başka bir sistemin alacağı JSON olarak yazdırır. `bundle["entry"][1]["resource"]` paketin ikinci girdisidir, yani Observation. `check_bundle` içinde `re.findall`, bir kaynağın metnindeki "reference" anahtarının bütün değerlerini bulur.

In [ ]:
final = fhir_bundle("VC-0128", result, model, digest, "2026-11-19T09:45:00+03:00", decision="confirmed", physician="dr-demo")
report = final["entry"][2]["resource"]
print("durum:", report["status"], "| raporu yapan:", report["performer"], "| kod:", report["conclusionCode"][0]["coding"][0]["display"])
print("Provenance etmenleri:", [a["type"]["text"] + " " + a["who"]["reference"] for a in final["entry"][3]["resource"]["agent"]])
print("sorunlar:", check_bundle(final))

**Gözlemle ve düşün.** Ön rapor kodlarında AF hakkında hiçbir şey ileri sürmez; 49436004 kodunu yalnızca kardiyoloğun kararı ekler. Kesin durumu kimler geri değiştirebilmeli ve bu olduğunda kayıt neyi saklamalıdır?

**Alıştırma G.** Kesin `final` paketindeki başvuruları, yani "reference" anahtarlarının değerlerini sayın. Kaç tane vardır?

In [ ]:
veri = alistirma_tablosu("G")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = len(re.findall(r'"reference": "([^"]+)"', json.dumps(final)))
teach.check("Alıştırma G", cevap)

## 8. Tahmin servisi ve prototipin dosyaları

*Ders: [Tahmin servisleri: Bir adresin arkasındaki model](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#prediction-services-the-model-behind-an-address)*

Bir tahmin servisi JSON biçimindeki isteklere cevap verir: İstek bir kaydın vuru zamanlarını, cevap ise modelin sürümü ve parmak iziyle birlikte sonucu taşır. Bu bölümün servisi yalnızca bu bilgisayarda, 127.0.0.1 adresinde dinler. Bölümün sonunda iki hücre prototipin dosyalarını yazar: Defterin fonksiyonlarını içeren `af_cdss.py` ve web arayüzünü içeren `app_af.py`. Ders sayfası arayüzü Streamlit bölümünde anlatmaktadır.

**Dersten koda.** Servisin cevabı: `predict`. Adres: `url`. İstek: `payload`. Cevap: `response`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
def predict(request, model, digest):
    """Tahmin servisi: Bir kaydın vuru zamanlarını içeren istek girer, sonuç çıkar; ikisi de JSON'un taşıyabileceği sözlüklerdir."""
    table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
    table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
    p = af_probability(model, table)
    alarm = p >= model["threshold"]
    runs = alarm_episodes(alarm)
    return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
            "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
            "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

answer_0128 = predict({"patient_id": "VC-0128", "beat_times_s": list(rec_0128["detected_s"])}, model, digest)
print({k: v for k, v in answer_0128.items() if k != "episodes"})

**Python notu.** `predict` bir sözlük, yani isteği alır ve bir sözlük, yani cevabı döndürür: İkisi de JSON olarak taşınabilir. İçeride defterle aynı fonksiyonları kullanır: `minutes_of`, `features`, `af_probability` ve `alarm_episodes`; böylece servis ve defter aynı sayıları hesaplar. `int(...)` ve `float(...)`, numpy sayılarını JSON'un yazabileceği sade Python sayılarına dönüştürür.

In [ ]:
import http.server
import threading
import urllib.request


class PredictionHandler(http.server.BaseHTTPRequestHandler):
    """POST /predict isteğini predict() sonucuyla JSON olarak yanıtlar."""

    def do_POST(self):
        body = self.rfile.read(int(self.headers["Content-Length"]))
        answer = json.dumps(predict(json.loads(body), model, digest)).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(answer)

    def log_message(self, *args):
        pass


server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{server.server_port}/predict"

rec_0128 = holter.recording()
payload = {"patient_id": "VC-0128", "beat_times_s": [round(t, 3) for t in rec_0128["detected_s"]]}
request = urllib.request.Request(url, data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"})
local = urllib.request.build_opener(urllib.request.ProxyHandler({}))   # istek bu bilgisayarda kalır, vekil sunucuya gitmez
response = json.loads(local.open(request, timeout=120).read().decode("utf-8"))
server.shutdown()

print(round(len(json.dumps(payload)) / 1024), "kB gönderildi:", url)
print({k: v for k, v in response.items() if k != "episodes"})
print(pd.DataFrame(response["episodes"]))

**Python notu.** `http.server.ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)` bu bilgisayarda küçük bir web sunucusu açar. 0 numaralı bağlantı noktası (port), işletim sisteminin boş bir bağlantı noktası seçmesini sağlar; seçilen noktayı `server.server_port` verir. `PredictionHandler` sınıfı her POST isteğini cevaplar: İsteğin gövdesini okur, `json.loads` ile bir sözlüğe dönüştürür, `predict` fonksiyonunu çağırır ve cevabı yazar. `threading.Thread(..., daemon=True).start()` sunucuyu defterin yanında çalıştırır; böylece sonraki satırlar ona bir istek gönderebilir. `urllib.request.Request` isteği baytlardan oluşan gövdesiyle kurar; `ProxyHandler({})` bir vekil sunucu tanımlı olsa bile isteğin bu bilgisayarda kalmasını sağlar; `server.shutdown()` de sonunda sunucuyu durdurur.

**Prototipin iki dosyası.** Sonraki iki hücre dosyaları `%%writefile` sihirli komutuyla yazar. Birincisi defterin fonksiyonlarını, ikincisi web arayüzünü tutar. Arayüzü başlatmak için Anaconda Prompt'u açın, `cd` komutuyla bu defterin klasörüne gidin ve `streamlit run app_af.py` komutunu çalıştırın. Tarayıcı sayfayı http://localhost:8501 adresinde açar; bu adres bilgisayarın kendisinde kalır. Arayüzü Anaconda Prompt'ta Ctrl+C ile durdurun.

In [ ]:
%%writefile af_cdss.py
"""af_cdss: Dersin AF karar desteğinin fonksiyonları; Jupyter defterinden bir dosyaya taşınmıştır (bir prototiptir, klinik kullanım için değildir)."""
import hashlib
import json
import pathlib
import re

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

WINDOW_S = 60


def minutes_of(beat_s, episodes=None, record=""):
    """Bir kaydın bir dakikalık pencereleri: Her dakikada biten RR aralıkları ve ritim biliniyorsa dakikanın AF içinde geçen payı."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows


def sample_entropy(rr, m=1, r=0.03):
    """Örneklem entropisi: İlk m aralığı birbirine benzeyen, yani en fazla r saniye farklı olan iki dizinin m + 1 aralıkta da benzer olma olasılığının logaritmasının eksi işaretlisi."""
    n = len(rr)

    def pairs(length):
        t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
        alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
        return (alike.sum() - len(t)) / 2

    b, a = pairs(m), pairs(m + 1)
    return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


def features(rr):
    """Bir dakikanın beş sayısı: Kalp hızı, değişim katsayısı, ortalama RR'ye bölünmüş RMSSD, pNN50 ve örneklem entropisi."""
    d = np.diff(rr)
    return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
            "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

V1_0 = ["hr", "cv", "rmssd", "pnn50"]
LOGGED = ["cv", "rmssd"]


def design(table, names):
    """Modelin değişkenleri: CV'nin ve RMSSD'nin logaritması, ötekiler olduğu gibi."""
    X = table[names].astype(float).copy()
    for name in names:
        if name in LOGGED:
            X[name] = np.log(X[name])
    return X


def fit_model(names, version, training):
    """Standartlaştırılmış değişkenlerle bir lojistik regresyon; JSON'un saklayabileceği düz bir sözlük olarak tutulur."""
    X = design(training, names)
    mean, sd = X.mean(), X.std()
    lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
    return {"name": "RR aralıklarından AF", "version": version, "features": names,
            "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
            "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


def af_probability(model, table):
    """Bir tablonun her dakikası için AF olasılığı."""
    names = model["features"]
    z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
    w = np.array([model["coef"][name] for name in names])
    return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


def choose_threshold(p, y, sensitivity=0.95):
    """AF dakikalarının istenen payını hâlâ bulan, 0,01 adımlı en yüksek eşik."""
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


def alarm_episodes(alarm):
    """Art arda gelen alarm dakikalarının dizileri: (ilk, son) konumları."""
    runs, first = [], None
    for k, a in enumerate(alarm):
        if a and first is None:
            first = k
        if not a and first is not None:
            runs.append((first, k - 1))
            first = None
    if first is not None:
        runs.append((first, len(alarm) - 1))
    return runs


def by_record(table, p, threshold):
    """Her kayıt için işaretlemenin ve modelin AF yükü, alarm atakları ve bunların yanlış olanları."""
    rows = []
    for record, part in table.assign(p=p).groupby("record", sort=True):
        alarm = (part["p"] >= threshold).to_numpy()
        runs = alarm_episodes(alarm)
        false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
        rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                     "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
    return pd.DataFrame(rows).set_index("record")


def contributions(model, row):
    """Her değişkenin bir dakikadaki AF log-odds değerine kattığı: Katsayısı çarpı z-skoru."""
    x = design(pd.DataFrame([row]), model["features"]).iloc[0]
    z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
    return {name: model["coef"][name] * z[name] for name in model["features"]}


def save_model(model, path):
    """Modeli JSON olarak yazar ve dosyanın parmak izi olan SHA-256 değerini döndürür."""
    text = json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False)
    pathlib.Path(path).write_text(text, encoding="utf-8")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def load_model(path):
    """save_model ile yazılmış bir modeli okur; modeli ve dosyanın SHA-256 değerini döndürür."""
    text = pathlib.Path(path).read_text(encoding="utf-8")
    return json.loads(text), hashlib.sha256(text.encode("utf-8")).hexdigest()

BASE_URL = "http://example.org/fhir"


def fhir_bundle(patient_id, result, model, digest, issued, decision=None, physician=None):
    """Bir kaydın raporu, HL7 FHIR R4 kaynakları olarak: Model bir Device, AF yükü bir Observation, ayrıca bir DiagnosticReport ve bir Provenance. Rapor, bir hekim onaylayana ya da reddedene kadar ön rapordur."""
    status = "final" if decision in ("confirmed", "rejected") else "preliminary"
    device = {"resourceType": "Device", "id": "af-rr",
              "deviceName": [{"name": model["name"], "type": "model-name"}],
              "version": [{"value": model["version"]}],
              "note": [{"text": "SHA-256 " + digest}]}
    burden = {"resourceType": "Observation", "id": "af-burden", "status": status,
              "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "procedure"}]}],
              "code": {"coding": [{"system": "http://example.org/vclinic", "code": "af-burden"}], "text": "AF yükü"},
              "subject": {"reference": "Patient/" + patient_id},
              "valueQuantity": {"value": result["burden_percent"], "unit": "%", "system": "http://unitsofmeasure.org", "code": "%"},
              "device": {"reference": "Device/af-rr"}}
    if result.get("start"):
        burden["effectivePeriod"] = {"start": result["start"], "end": result["end"]}
    report = {"resourceType": "DiagnosticReport", "id": "af-report", "status": status,
              "code": {"text": "Holter kaydı, AF analizi"},
              "subject": {"reference": "Patient/" + patient_id}, "issued": issued,
              "result": [{"reference": "Observation/af-burden"}],
              "conclusion": result["text"]}
    agents = [{"type": {"text": "author"}, "who": {"reference": "Device/af-rr"}}]
    if status == "final":
        report["performer"] = [{"reference": "Practitioner/" + physician}]
        agents.append({"type": {"text": "verifier"}, "who": {"reference": "Practitioner/" + physician}})
        if decision == "confirmed" and result["burden_percent"] > 0:
            report["conclusionCode"] = [{"coding": [{"system": "http://snomed.info/sct", "code": "49436004",
                                                      "display": "Atrial fibrillation (disorder)"}]}]
    provenance = {"resourceType": "Provenance", "id": "af-provenance", "target": [{"reference": "DiagnosticReport/af-report"}],
                  "recorded": issued, "agent": agents}
    resources = [device, burden, report, provenance]
    return {"resourceType": "Bundle", "type": "collection",
            "entry": [{"fullUrl": f"{BASE_URL}/{r['resourceType']}/{r['id']}", "resource": r} for r in resources]}


def check_bundle(bundle):
    """Bir paketin basit denetimleri: Her kaynağın bir türü ve bir kimliği vardır, paketin içindeki her başvuru bir kaynağa ulaşır ve yükün bir UCUM birimi vardır. Patient ve Practitioner hastanenin sisteminde bulunur."""
    problems = []
    present = {f"{e['resource']['resourceType']}/{e['resource'].get('id')}" for e in bundle["entry"]}
    for e in bundle["entry"]:
        r = e["resource"]
        if not r.get("id"):
            problems.append(r["resourceType"] + ": kimlik yok")
        for ref in re.findall(r'"reference": "([^"]+)"', json.dumps(r)):
            if ref.split("/")[0] not in ("Patient", "Practitioner") and ref not in present:
                problems.append("başvuru bir kaynağa ulaşmıyor: " + ref)
        if r["resourceType"] == "Observation":
            q = r.get("valueQuantity", {})
            if q.get("system") != "http://unitsofmeasure.org":
                problems.append("Observation: birim UCUM değil")
    return problems


def predict(request, model, digest):
    """Tahmin servisi: Bir kaydın vuru zamanlarını içeren istek girer, sonuç çıkar; ikisi de JSON'un taşıyabileceği sözlüklerdir."""
    table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
    table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
    p = af_probability(model, table)
    alarm = p >= model["threshold"]
    runs = alarm_episodes(alarm)
    return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
            "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
            "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

In [ ]:
%%writefile app_af.py
"""AF karar desteği: Dersin prototipinin web arayüzü (klinik kullanım için değildir). Anaconda Prompt'ta şu komutla başlatın: streamlit run app_af.py"""
import datetime
import json
import pathlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import streamlit as st

HERE = pathlib.Path(__file__).resolve().parent
for folder in [HERE, *HERE.parents]:
    if (folder / "vclinic" / "__init__.py").exists():
        sys.path.insert(0, str(folder))
        break
sys.path.insert(0, str(HERE))
import af_cdss as cdss
from vclinic import afdb, holter

st.set_page_config(page_title="AF karar desteği", layout="wide")
st.title("AF karar desteği")
st.caption("Sentetik ve herkese açık verilerle kurulmuş bir ders prototipidir; klinik kullanım için değildir.")

files = sorted(HERE.glob("af_model_v*.json"))
if not files:
    st.error("Uygulamanın yanında model dosyası yok. Önce Jupyter defterinin 6. bölümünü çalıştırın.")
    st.stop()
chosen = st.sidebar.selectbox("Model dosyası", [f.name for f in files], index=len(files) - 1)
model, digest = cdss.load_model(HERE / chosen)
st.sidebar.write(f"Sürüm {model['version']} · eşik {model['threshold']:.2f} · SHA-256 {digest[:12]}")


@st.cache_data
def ward_recordings():
    return holter.ward()


ward = ward_recordings()
sources = {f"Servis {pid}": ("ward", pid) for pid in ward}
sources.update({f"AFDB {r}": ("afdb", r) for r in afdb.RECORDS if (afdb.default_folder() / f"{r}.qrs").exists()})
label = st.sidebar.selectbox("Kayıt", list(sources))
kind, rid = sources[label]
beat_s = ward[rid]["detected_s"] if kind == "ward" else afdb.beats(rid)

table = pd.DataFrame(cdss.minutes_of(beat_s))
table = table.join(pd.DataFrame([cdss.features(x) for x in table["rr"]], index=table.index))
p = cdss.af_probability(model, table)
alarm = p >= model["threshold"]
runs = cdss.alarm_episodes(alarm)
hours = (table["start_s"] / 3600).round(3)

c1, c2, c3 = st.columns(3)
c1.metric("AF yükü", f"{100 * alarm.mean():.1f} %")
c2.metric("Alarm atakları", len(runs))
c3.metric("İncelenen dakika", len(table))
st.line_chart(pd.DataFrame({"AF olasılığı": p, "eşik": model["threshold"]}, index=hours))

if runs:
    episodes = pd.DataFrame([{"başlangıç (sa)": hours.iloc[a], "dakika": b - a + 1,
                              "en yüksek olasılık": round(float(p[a:b + 1].max()), 2)} for a, b in runs])
    st.dataframe(episodes)
    k = st.selectbox("İncelenecek alarm atağı", list(range(len(runs))),
                     format_func=lambda i: f"{i + 1}: {episodes.iloc[i, 0]:.2f} sa")
    row = table.iloc[runs[k][0]]
    left, right = st.columns(2)
    with left:
        st.write("Atağın ilk dakikasında her değişkenin AF log-odds değerine kattığı")
        st.bar_chart(pd.Series(cdss.contributions(model, row), name="log-odds"))
    with right:
        fig, ax = plt.subplots(figsize=(6, 2.6))
        if kind == "afdb" and (afdb.default_folder() / f"{rid}.dat").exists():
            t, x = afdb.strip(rid, row["start_s"], 10)
            ax.plot(t - t[0], x[:, 0], lw=0.7)
            ax.set_title(f"AFDB {rid} · MIT-BIH AFDB, PhysioNet, ODC-By 1.0", fontsize=8)
        elif kind == "ward":
            t, x = holter.strip(ward[rid], row["start_s"])
            ax.plot(t, x, lw=0.7)
            ax.set_title(f"{rid} · DII, sentetik", fontsize=8)
        else:
            ax.plot(row["rr"], marker=".", lw=0.7)
            ax.set_title("Dakikanın RR aralıkları (s); sinyal dosyası yok", fontsize=8)
        ax.set_xlabel("s")
        st.pyplot(fig)
    review = f"alarm atağı {k + 1}"
else:
    review = "alarm yok"

st.subheader("Hekimin kararı")
decision = st.radio("Prototipin AF bulgusu", ["incelenmedi", "onaylandı", "reddedildi"])
reason = st.text_input("Gerekçe (örneğin erken vurular, gürültü)")
physician = st.text_input("Hekim", value="dr-demo")
code = {"incelenmedi": None, "onaylandı": "confirmed", "reddedildi": "rejected"}[decision]
now = datetime.datetime.now().astimezone().isoformat(timespec="seconds")
if st.button("Kararı kayda yaz"):
    log = HERE / "decision_log.csv"
    entry = {"time": now, "recording": label, "review": review, "model_version": model["version"], "sha256": digest,
             "ai_burden_percent": round(100 * float(alarm.mean()), 1), "decision": code or "not reviewed", "reason": reason, "physician": physician}
    pd.DataFrame([entry]).to_csv(log, mode="a", header=not log.exists(), index=False)
    st.success("Şu dosyaya yazıldı: " + log.name)

period = ("2026-11-18T08:00:00+03:00", "2026-11-19T08:00:00+03:00") if kind == "ward" else (None, None)
result = {"start": period[0], "end": period[1], "burden_percent": round(100 * float(alarm.mean()), 1),
          "text": f"AF yükü {100 * alarm.mean():.1f} %, {len(runs)} alarm atağı; model {model['version']}."}
bundle = cdss.fhir_bundle(rid, result, model, digest, now, code, physician)
st.download_button("HL7 FHIR raporu (JSON)", json.dumps(bundle, indent=2, ensure_ascii=False),
                   file_name=f"report_{rid}.json", mime="application/json")

In [ ]:
import importlib
import af_cdss
importlib.reload(af_cdss)
same = np.allclose(af_cdss.af_probability(model, validation), p_val)
print("af_cdss defterle aynı olasılıkları veriyor:", same)
for name in ("af_cdss.py", "app_af.py"):
    print(name, pathlib.Path(name).stat().st_size, "bayt")

**Gözlemle ve düşün.** Arayüz modeli bir dosyadan okur ve parmak izini yan çubukta gösterir. Bir vardiya sırasında biri dosyayı yeni bir sürümle değiştirdiğinde serviste ne olmalıdır?

**Alıştırma H.** Servis VC-0128'in 24 saatinde kaç dakikayı inceledi?

In [ ]:
veri = alistirma_tablosu("H")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(response["minutes"])
teach.check("Alıştırma H", cevap)

## 9. Telemetri servisinde kullanıcı testi

*Ders: [Kullanıcı testi: Kullanıcıları iş başında izlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#the-user-test-watching-users-at-work)*

Sanal kliniğin telemetri servisinde dört hasta vardır: Paroksismal AF'si, atriyal erken vuruları ve iki kısa gürültü dönemi olan VC-0128; sık atriyal erken vuruları olan ve AF'si olmayan VC-0347; bütün gün AF'de olan VC-0519; uzun elektrot gürültüsü dönemleri olan ve AF'si olmayan VC-0733. Kayıtları ders paketinden gelir; prototip kayıt cihazının bulduğu vuruları görür. Bu bölüm hekimin 24 saatte inceleyeceklerini sayar ve bir karar kaydı yazar.

**Dersten koda.** Servisin dakikaları: `ward_minutes`. Her hastanın alarm atakları: `by_record`. Karar kaydı: `log`; `decision_log_ward.csv` olarak kaydedilir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
ward = holter.ward()
ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                             for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))

ward_minutes["p10"] = af_probability(model, ward_minutes)
review = by_record(ward_minutes, ward_minutes["p10"].to_numpy(), model["threshold"])
print(review.round(1))
print("24 saatte incelenecek:", int(review["episodes"].sum()), "atak; yanlış:", int(review["false_episodes"].sum()))

**Python notu.** `holter.ward()` her hastanın kimliğinden kaydına giden bir sözlük döndürür. `ward_minutes` içindeki liste üreteci her kaydı aynı `minutes_of` fonksiyonuyla dakikalara böler ve gerçek olarak kaydın AF ataklarını kullanır; derste bu ataklar bir kardiyoloğun EKG şeritlerini okumasının yerini tutar. `by_record`, "record" sütunu olan her tabloyla çalışır; burada bu sütun hastaları tutar.

In [ ]:
rows = []
for pid, part in ward_minutes.groupby("record"):
    alarm = (part["p10"] >= model["threshold"]).to_numpy()
    for a, b in alarm_episodes(alarm):
        true_af = int(part["af"].to_numpy()[a:b + 1].sum() > 0)
        rows.append({"hasta": pid, "başlangıç (sa)": round(part["start_s"].iloc[a] / 3600, 2), "dakika": b - a + 1,
                     "karar": "onaylandı" if true_af else "reddedildi",
                     "gerekçe": "P dalgası olmadan düzensiz RR" if true_af else
                                         ("elektrot gürültüsü" if pid == "VC-0733" else "atriyal erken vurular"),
                     "sürüm": model["version"], "SHA-256": digest[:12]})
log = pd.DataFrame(rows)
log.to_csv("decision_log_ward.csv", index=False)
print(log.groupby(["hasta", "karar"]).size())
print(log.head(8).to_string(index=False))

**Python notu.** Döngü her alarm atağı için karar kaydına bir satır yazar: Hasta, başlangıç, uzunluk, karar, gerekçesi ve modelin sürümü ile parmak izi. Burada kararlar sentetik kayıtların gerçeğini izler; böylece kayıt dikkatli bir hekimin vereceği kararları gösterir. Bir serviste kaydı hekim doldurur. `to_csv(..., index=False)` tabloyu bir hesap tablosunun açabildiği, virgülle ayrılmış değerlerden oluşan bir dosya olarak kaydeder; `groupby([...]).size()` de her hasta ve karar için satırları sayar.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9.6, 7.0), sharex=True)
for ax, (pid, part) in zip(axes, ward_minutes.groupby("record")):
    hours_ = part["start_s"] / 3600
    ax.fill_between(hours_, 0, part["af"], color="#C8C0B8", step="post")
    ax.plot(hours_, part["p10"], color="#9B1B1B", lw=0.7)
    ax.axhline(model["threshold"], color="#1F5F8B", ls="--", lw=0.8)
    ax.set_ylabel(pid, fontsize=9)
    ax.set_ylim(0, 1.02)
axes[-1].set_xlabel("08.00'den itibaren saat")
axes[0].set_title("Telemetri servisinde 1.0 sürümünün AF olasılığı (gri: kayıttaki AF; sentetik veriler)", fontsize=10)
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** VC-0347 öğleden sonra ve akşam çok sayıda kısa yanlış alarm, VC-0733 ise gürültü dönemlerinde birkaç yanlış alarm verir. Bu iki nedenden hangisini dakikayı daha iyi tarif eden bir sayı ortadan kaldırabilir ve hangisi bunun yerine sinyalin denetlenmesini gerektirir?

**Alıştırma I.** 1.0 sürümü VC-0347 için 24 saatte kaç yanlış alarm atağı verir?

In [ ]:
veri = alistirma_tablosu("I")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(review.loc["VC-0347", "false_episodes"])
teach.check("Alıştırma I", cevap)

## 10. 1.1 sürümü ve iki sürümün testi

*Ders: [Prototipi iyileştirmek: 1.1 sürümü](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html#improving-the-prototype-version-1-1)*

1.1 sürümü dört sayıya örneklem entropisini ekler, eğitim kayıtlarında yeniden öğrenir ve doğrulama kayıtlarında kendi eşiğini alır. Test kayıtları ancak bundan sonra, her sürüm için bir kez açılır; değişiklik kaydı da neyin iyileştiğini ve bunun neye mal olduğunu bildirir.

**Dersten koda.** 1.1 sürümünün beş sayısı: `V1_1`. Model: `model_11`. Testteki karşılaştırma: `compare`. Değişiklik kaydı: `changes`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB10_klinik_karar_destegi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-10/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
V1_1 = V1_0 + ["sampen"]
model_11 = fit_model(V1_1, "1.1.0", training)
p_val_11 = af_probability(model_11, validation)
model_11["threshold"] = choose_threshold(p_val_11, validation["af"].to_numpy())

records_11 = by_record(validation, p_val_11, model_11["threshold"])
print("eşik", model_11["threshold"], "| doğrulama AUC", round(roc_auc_score(validation["af"], p_val_11), 3))
print("doğrulama kayıtlarında yanlış alarm atakları: 1.0 sürümü", int(records_val["false_episodes"].sum()),
      "| 1.1 sürümü", int(records_11["false_episodes"].sum()))
print("08219 kaydının yanlış alarm dakikaları:", int(((p_val >= model["threshold"]) & (validation["af"] == 0) & (validation["record"] == "08219")).sum()),
      "→", int(((p_val_11 >= model_11["threshold"]) & (validation["af"] == 0) & (validation["record"] == "08219")).sum()))

**Python notu.** `V1_0 + ["sampen"]` beş addan oluşan yeni bir liste oluşturur; 1.0 sürümünün listesi olduğu gibi kalır. `fit_model` ve `choose_threshold`, 4. bölümün başka argümanlarla çağrılan fonksiyonlarıdır: Bir sürümün kodu aynıdır, yalnızca girdileri değişir. Son satır `&` ile birleştirilmiş üç maskeyle 08219 kaydında AF olmadığı hâlde her sürümde alarm veren dakikaları sayar.

In [ ]:
rows = []
for name, mdl in (("1.0", model), ("1.1", model_11)):
    p_te = af_probability(mdl, test)
    alarm = p_te >= mdl["threshold"]
    rec = by_record(test, p_te, mdl["threshold"])
    rows.append({"sürüm": name, "AUC": round(roc_auc_score(test["af"], p_te), 3),
                 "duyarlılık": round(alarm[test["af"].to_numpy() == 1].mean(), 3),
                 "özgüllük": round(1 - alarm[test["af"].to_numpy() == 0].mean(), 3),
                 "yanlış alarm atakları": int(rec["false_episodes"].sum()),
                 "AF yükündeki hata (puan)": round(float((rec["model_burden"] - rec["burden"]).abs().mean()), 1)})
compare = pd.DataFrame(rows)
print(compare.to_string(index=False))
digest_11 = save_model(model_11, "af_model_v1.1.0.json")
changes = {"sürüm": "1.1.0", "değişiklik": "beşinci sayı olarak örneklem entropisi",
           "eşik": model_11["threshold"], "test: duyarlılık": f"{compare.iloc[0, 2]} → {compare.iloc[1, 2]}",
           "test: yanlış alarm atakları": f"{compare.iloc[0, 4]} → {compare.iloc[1, 4]}", "SHA-256": digest_11[:12]}
print(changes)

**Python notu.** Döngü iki sürümü test kayıtlarında aynı kodla değerlendirir. `alarm[test["af"].to_numpy() == 1].mean()` alarm veren AF dakikalarının payıdır, yani duyarlılıktır; bir eksi öteki dakikalardan alarm verenlerin payı da özgüllüktür. `compare.iloc[0, 2]` ilk satırın üçüncü sütunundaki hücreyi okur. `save_model` 1.1 sürümünün dosyasını yazar; web arayüzü bu dosyayı 1.0 sürümünün dosyasının yanında listeler.

**Gözlemle ve düşün.** 1.1 sürümü test kayıtlarında daha az yanlış alarm verir, ancak daha fazla AF dakikası kaçırır. Her AF atağını antikoagülasyonla tedavi eden bir servis için hangi sürümü, alarmlara boğulan bir servis için ise hangisini seçerdiniz?

**İleri düzey (isteğe bağlı).** VC-0733'ün gürültüsü iki sürümü de yanıltır. Sinyal kalitesinin denetlenmesi, örneğin bir aralığın 0,25 saniyeden kısa ya da komşularından çok uzun olduğu dakikalar gibi kaçırılan ya da eklenen vuruları olan dakikaları işaretleyebilir ve bunları alarmın dışında bırakabilir. Böyle bir kuralı `ward_minutes` üzerinde deneyin ve VC-0733'ün yanlış alarm ataklarını yeniden sayın.

**Alıştırma J.** 1.1 sürümünün test kayıtlarındaki duyarlılığı nedir? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("J")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = float(compare.iloc[1]["duyarlılık"])
teach.check("Alıştırma J", cevap)

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Telemetri ve koroner bakım | Servisinizde bir AF alarmının iş akışını yazın: Alarmı kim görür, ne kadar sürede, hangi ekranda ve karar kaydı neyi tutmalıdır? 9. bölümde 1.0 sürümünün hasta ve gün başına alarm ataklarını sayın ve hemşirelerinizin bunları inceleyip inceleyemeyeceğine karar verin. |
| Elektrofizyoloji | Bir ablasyondan sonra bir Holter kaydının AF yükü bir sonlanımdır. Her test kaydının yükünü iki sürümle hesaplayın ve 1.1 sürümünün hatasının bir hastayı ablasyondan önce ve sonra karşılaştırmaya yetecek kadar küçük olup olmadığını değerlendirin. |
| İnmeden korunma | Kaçırılan bir AF atağı kaçırılmış bir antikoagülasyon anlamına gelebilir. Doğrulama kayıtlarında 0,99 duyarlılığı koruyan 1.0 sürümü eşiğini bulun ve bu eşiğin yanlış alarm ataklarına ne yaptığını bildirin. |
| Kalp yetersizliği polikliniği | Kalp yetersizliği olan hastalarda erken vurular sıktır. 9. bölümü kullanarak VC-0347'nin erken vurulu olan ve olmayan saatleri arasında dakikanın hangi sayılarının en çok değiştiğini listeleyin ve sık ektopisi olan hastalar için model kartına bir cümle önerin. |
| Giyilebilir cihazlar ve uzaktan izlem | Bir saat R tepelerinin değil, nabzın zamanlarını verir ve daha fazla vuru kaçırır. VC-0128'in vurularından rastgele yirmide birini çıkarın, `predict` fonksiyonunu yeniden çalıştırın ve AF yükünü kaydın tamamınınkiyle karşılaştırın. |
| Kalite ve hasta güvenliği | 9. bölümün karar kaydını bir izleme aracı olarak kullanın: Hasta başına reddedilen alarm ataklarının payını hesaplayın ve bu pay yükseldiğinde kliniğin modelin sürümünü gözden geçirmesini sağlayacak kuralı yazın. |
| Araştırma ve etik | Prototipin servisinizde sınanacağı bir çalışma için bir etik kurulun ihtiyaç duyacağı yarım sayfayı yazın: Kullanım amacı, veriler ve verilerin nerede kaldığı, hastaların bilgilendirilmesi, hekimin kararı ve yanlış alarmlar için plan. |

## Defterden sonra

Bu haftanın prototipi bir monitörün vurularını okur, alarmlarını açıklar, başka sistemlerin okuyabileceği bir rapor yazar ve her çıktısında modelinin sürümünü tutar. Haftalık proje prototipi kullanıcıları üzerinde sınar ve iyileştirir. 11. hafta final projesine geçer: Katılımcılar kendi kliniklerinden bir soru için veriden modele, modelin açıklamasına ve klinik değerlendirmesinin tasarımına uzanan bir çalışma hazırlar.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>